# TAHAP 09 — Requirement & Taxonomy Redesign for Production-Ready v2

**Catatan penting:** tahap ini belum bertujuan melatih ulang model. Output utama tahap ini adalah rancangan taxonomy, requirement, data dictionary, skema dataset v2, dan konfigurasi awal scoring engine untuk tahap berikutnya.

## 1. Tujuan Notebook

Notebook ini disusun untuk menghasilkan:

1. Business understanding berbasis CRISP-DM.
2. Gap analysis dari chatbot v1 menuju production-ready v2.
3. Functional requirement dan non-functional requirement.
4. Redesign user persona.
5. Redesign taxonomy sekolah Indonesia.
6. Redesign taxonomy program studi Indonesia dengan gaya pemetaan PDDikti.
7. Redesign taxonomy profil siswa: RIASEC, Multiple Intelligences, VARK, Grit/Mindset, dan Career Alignment.
8. Data dictionary seluruh taxonomy.
9. Rancangan struktur dataset v2.
10. Rancangan scoring engine dan recommender logic awal.
11. Rancangan output chatbot production-ready.
12. Kesimpulan akademik dan checklist output untuk tahap berikutnya.

In [1]:
# 1. Setup Environment dan Path Project

from pathlib import Path
import json
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 120)

# Deteksi project root secara fleksibel.
# Aman dijalankan dari root project, dari folder notebooks/, atau dari ZIP yang memiliki subfolder project.
CURRENT_DIR = Path.cwd().resolve()
ROOT_CANDIDATES = [
    CURRENT_DIR,
    CURRENT_DIR.parent,
    CURRENT_DIR / 'edupath-career-assitant',
    CURRENT_DIR.parent / 'edupath-career-assitant',
    CURRENT_DIR / 'edupath-career-assistant',
    CURRENT_DIR.parent / 'edupath-career-assistant',
]

# Prioritaskan root yang memiliki reports/models/notebooks karena notebook tahap 09 ditempatkan di root project.
PROJECT_ROOT = None
for c in ROOT_CANDIDATES:
    if (c / 'reports').exists() or (c / 'models').exists() or (c / 'notebooks').exists():
        PROJECT_ROOT = c
        break
if PROJECT_ROOT is None:
    PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR

# DATA_PROCESSED_DIR dipilih dari lokasi yang benar-benar memiliki dataset processed v1.
DATA_DIR_CANDIDATES = [
    PROJECT_ROOT / 'data' / 'processed',
    PROJECT_ROOT / 'edupath-career-assitant' / 'data' / 'processed',
    PROJECT_ROOT / 'edupath-career-assistant' / 'data' / 'processed',
    CURRENT_DIR / 'data' / 'processed',
    CURRENT_DIR.parent / 'data' / 'processed',
    CURRENT_DIR / 'edupath-career-assitant' / 'data' / 'processed',
    CURRENT_DIR.parent / 'edupath-career-assitant' / 'data' / 'processed',
]

DATA_PROCESSED_DIR = None
for d in DATA_DIR_CANDIDATES:
    if (d / 'intent_dataset_processed.csv').exists() or (d / 'program_studi_s1_processed.csv').exists():
        DATA_PROCESSED_DIR = d
        break
if DATA_PROCESSED_DIR is None:
    DATA_PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

# Output taxonomy v2 ditempatkan di PROJECT_ROOT/data/taxonomy agar rapi untuk tahap berikutnya.
TAXONOMY_DIR = PROJECT_ROOT / 'data' / 'taxonomy'
REPORT_STAGE09_DIR = PROJECT_ROOT / 'reports' / 'stage09'
MODEL_DIR = PROJECT_ROOT / 'models'

for d in [DATA_PROCESSED_DIR, TAXONOMY_DIR, REPORT_STAGE09_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('CURRENT_DIR      :', CURRENT_DIR)
print('PROJECT_ROOT     :', PROJECT_ROOT)
print('DATA_PROCESSED   :', DATA_PROCESSED_DIR)
print('TAXONOMY_DIR     :', TAXONOMY_DIR)
print('REPORT_STAGE09   :', REPORT_STAGE09_DIR)
print('MODEL_DIR        :', MODEL_DIR)

CURRENT_DIR      : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\notebooks
PROJECT_ROOT     : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant
DATA_PROCESSED   : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\data\processed
TAXONOMY_DIR     : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\data\taxonomy
REPORT_STAGE09   : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\reports\stage09
MODEL_DIR        : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\models


## 2. Business Understanding Berbasis CRISP-DM

### 2.1 Business Context

EduPath Career Assistant ditujukan untuk membantu siswa kelas akhir jenjang pendidikan menengah Indonesia, terutama SMA, MA, SMK, dan MAK, dalam memperoleh rekomendasi awal program studi S1 beserta alasan, roadmap belajar, dan prospek karier.

Pada versi v1, sistem sudah dapat melakukan intent classification, intent routing, rekomendasi program studi, roadmap, skill awal, dan template respons. Namun, hasil evaluasi dan uji aplikasi menunjukkan bahwa sistem masih bersifat baseline akademik karena:

- dataset masih kecil;
- taxonomy program studi belum cukup luas;
- latar belakang siswa masih direpresentasikan sebagai `kelas`, bukan konteks sekolah dan jurusan;
- sistem belum menggunakan kerangka minat, kecerdasan, gaya belajar, ketahanan belajar, dan career alignment secara eksplisit;
- sistem belum membandingkan pilihan intuisi awal siswa dengan hasil pemetaan profil.

### 2.2 Business Objective

Membangun desain v2 yang lebih **production-oriented**, yaitu sistem yang memiliki standardisasi input, taxonomy eksplisit, data dictionary, rancangan synthetic data generator, dan scoring engine yang transparan.

### 2.3 Data Mining Objective

Tujuan data mining pada tahap v2 adalah:

1. Mengklasifikasikan intent percakapan siswa secara lebih stabil.
2. Mengekstraksi profil siswa dari teks percakapan dan input terstruktur.
3. Memetakan profil siswa ke taxonomy RIASEC, Multiple Intelligences, VARK, Grit/Mindset, dan Career Alignment.
4. Menghasilkan Top-N rekomendasi program studi S1 berbasis hybrid scoring.
5. Menyediakan alasan rekomendasi yang dapat dijelaskan secara akademik dan dapat divalidasi oleh guru BK, orang tua, atau pembimbing.

### 2.4 Success Criteria

| Dimensi | Target v2 |
|---|---|
| Dataset intent | minimal 10.000–15.000 synthetic utterances, seimbang per intent |
| Program studi | minimal 30–50 prodi S1 tahap awal, bisa diperluas bertahap |
| Taxonomy sekolah | mendukung SMA/MA/SMK/MAK dan jurusan/konsentrasi siswa |
| Profil psikopedagogik | RIASEC, MI, VARK, Grit/Mindset, Career Alignment |
| Explainability | setiap rekomendasi memiliki alasan berbasis score component |
| Safety | output berupa rekomendasi awal, bukan keputusan final atau diagnosis psikologis |
| Evaluation | intent accuracy, macro F1, top-k recommendation accuracy, scenario pass rate |

## 3. Data Understanding Awal dari Project v1

Cell berikut membaca artifact v1 jika tersedia. Tujuannya bukan untuk melatih model, melainkan untuk memahami baseline existing sebelum redesign.

In [2]:
# 2. Membaca Inventory Dataset dan Report v1 Jika Tersedia

def read_csv_if_exists(path: Path):
    if path.exists():
        return pd.read_csv(path)
    return None

v1_files = {
    'intent_dataset_processed': DATA_PROCESSED_DIR / 'intent_dataset_processed.csv',
    'program_studi_s1_processed': DATA_PROCESSED_DIR / 'program_studi_s1_processed.csv',
    'program_recommender_profiles_stage05': DATA_PROCESSED_DIR / 'program_recommender_profiles_stage05.csv',
    'chatbot_evaluation_metrics_stage07': PROJECT_ROOT / 'reports' / 'stage07' / 'chatbot_evaluation_metrics_stage07.csv',
    'chatbot_intent_summary_stage07': PROJECT_ROOT / 'reports' / 'stage07' / 'chatbot_intent_summary_stage07.csv',
    'chatbot_error_analysis_stage07': PROJECT_ROOT / 'reports' / 'stage07' / 'chatbot_error_analysis_stage07.csv',
    'model_comparison_stage04': PROJECT_ROOT / 'reports' / 'modeling' / 'model_comparison_stage04.csv',
}

inventory_rows = []
loaded_data = {}

for name, path in v1_files.items():
    df = read_csv_if_exists(path)
    loaded_data[name] = df
    inventory_rows.append({
        'artifact_name': name,
        'path': str(path.relative_to(PROJECT_ROOT)) if path.exists() else str(path),
        'exists': path.exists(),
        'rows': 0 if df is None else len(df),
        'columns': 0 if df is None else len(df.columns),
        'column_list': '' if df is None else ', '.join(df.columns.astype(str).tolist())
    })

inventory_df = pd.DataFrame(inventory_rows)
inventory_output = REPORT_STAGE09_DIR / 'current_project_inventory_stage09.csv'
inventory_df.to_csv(inventory_output, index=False)

display(inventory_df)
print('Saved:', inventory_output)

,artifact_name,path,exists,rows,columns,column_list
0,intent_dataset_processed,data\processed\intent_dataset_processed.csv,True,12,21,"utterance_id, utterance, intent_label, bahasa, register, jenjang_user, kelas, minat_keywords, mapel_favorit, hobi_ke..."
1,program_studi_s1_processed,data\processed\program_studi_s1_processed.csv,True,5,22,"program_id, nama_program_studi, jenjang, rumpun_ilmu, bidang_keilmuan, deskripsi_singkat, minat_cocok, mapel_relevan..."
2,program_recommender_profiles_stage05,data\processed\program_recommender_profiles_stage05.csv,True,5,25,"program_id, nama_program_studi, jenjang, rumpun_ilmu, bidang_keilmuan, deskripsi_singkat, minat_cocok, mapel_relevan..."
3,chatbot_evaluation_metrics_stage07,reports\stage07\chatbot_evaluation_metrics_stage07.csv,True,7,2,"metric, value"
4,chatbot_intent_summary_stage07,reports\stage07\chatbot_intent_summary_stage07.csv,True,8,5,"expected_intent, total_cases, correct_intent, intent_accuracy, overall_pass_rate"
5,chatbot_error_analysis_stage07,reports\stage07\chatbot_error_analysis_stage07.csv,True,4,37,"test_id, scenario_group, user_input, expected_intent, final_intent, intent_correct, routing_source, model_predicted_..."
6,model_comparison_stage04,reports\modeling\model_comparison_stage04.csv,True,3,6,"model, accuracy, precision_weighted, recall_weighted, f1_weighted, f1_macro"


Saved: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\reports\stage09\current_project_inventory_stage09.csv


## 4. Gap Analysis dari Chatbot v1 ke Production-Ready v2

Gap analysis dilakukan untuk mengidentifikasi perbedaan antara kondisi existing dengan kebutuhan production-oriented v2. Fokus gap tidak hanya pada model, tetapi juga pada requirement, taxonomy, data quality, explainability, dan governance.

In [3]:
# 3. Gap Analysis v1 ke v2

# Ambil angka baseline jika data tersedia.
intent_df = loaded_data.get('intent_dataset_processed')
program_df = loaded_data.get('program_studi_s1_processed')
metrics_df = loaded_data.get('chatbot_evaluation_metrics_stage07')
intent_summary_df = loaded_data.get('chatbot_intent_summary_stage07')
error_df = loaded_data.get('chatbot_error_analysis_stage07')

intent_rows = 0 if intent_df is None else len(intent_df)
intent_count = 0 if intent_df is None or 'intent_label' not in intent_df.columns else intent_df['intent_label'].nunique()
program_rows = 0 if program_df is None else len(program_df)

stage07_overall_pass = None
if metrics_df is not None:
    # Mendukung format metrik yang berbeda-beda.
    try:
        if 'metric' in metrics_df.columns and 'value' in metrics_df.columns:
            m = dict(zip(metrics_df['metric'], metrics_df['value']))
            stage07_overall_pass = m.get('overall_pass_rate')
        elif 'overall_pass_rate' in metrics_df.columns:
            stage07_overall_pass = metrics_df['overall_pass_rate'].iloc[0]
    except Exception:
        stage07_overall_pass = None

klarifikasi_accuracy = None
if intent_summary_df is not None and 'expected_intent' in intent_summary_df.columns:
    row = intent_summary_df[intent_summary_df['expected_intent'].astype(str).str.contains('klarifikasi_minat', case=False, na=False)]
    if len(row) > 0 and 'intent_accuracy' in row.columns:
        klarifikasi_accuracy = row['intent_accuracy'].iloc[0]

error_count = 0 if error_df is None else len(error_df)

gap_analysis = pd.DataFrame([
    {
        'area': 'Dataset Intent',
        'condition_v1': f'{intent_rows} utterance, {intent_count} intent',
        'gap': 'Belum cukup untuk model intent yang robust dan generalizable.',
        'target_v2': '10.000–15.000 synthetic utterances dengan variasi bahasa formal, non-formal, Jawa umum, Sunda umum, dan off-domain.',
        'priority': 'High',
        'stage10_action': 'Bangun synthetic data generator dan balanced scenario sampling.'
    },
    {
        'area': 'Taxonomy Program Studi',
        'condition_v1': f'{program_rows} program studi S1',
        'gap': 'Cakupan prodi terlalu sempit sehingga rekomendasi bias ke beberapa prodi awal.',
        'target_v2': 'Minimal 30–50 prodi S1 PDDikti-style dengan rumpun/bidang ilmu dan profil kecocokan.',
        'priority': 'High',
        'stage10_action': 'Perluas katalog prodi dan validasi nama prodi terhadap referensi PDDikti.'
    },
    {
        'area': 'Input Profil Siswa',
        'condition_v1': 'Field kelas masih muncul sebagai variabel utama.',
        'gap': 'Tidak merepresentasikan konteks Indonesia seperti SMA/MA/SMK/MAK dan jurusan/konsentrasi siswa.',
        'target_v2': 'Gunakan school_type, school_major_group, school_major_detail, dan subject_strength.',
        'priority': 'High',
        'stage10_action': 'Ganti field kelas menjadi taxonomy sekolah dan jurusan asal siswa.'
    },
    {
        'area': 'Psychopedagogical Mapping',
        'condition_v1': 'Minat, hobi, mapel, dan gaya belajar masih berupa keyword bebas.',
        'gap': 'Belum ada standardisasi RIASEC, Multiple Intelligences, VARK, Grit/Mindset, dan Career Alignment.',
        'target_v2': 'Semua profil siswa dipetakan ke taxonomy eksplisit dan score component.',
        'priority': 'High',
        'stage10_action': 'Bangun taxonomy CSV dan extractor logic.'
    },
    {
        'area': 'Intent Routing dan Klarifikasi',
        'condition_v1': f'Akurasi klarifikasi_minat: {klarifikasi_accuracy if klarifikasi_accuracy is not None else "N/A"}',
        'gap': 'Input siswa yang ragu/bingung masih dapat diarahkan langsung ke rekomendasi.',
        'target_v2': 'Intent klarifikasi_minat harus memicu guided profiling, bukan langsung Top-N prodi.',
        'priority': 'High',
        'stage10_action': 'Tambahkan guided question flow dan rule untuk uncertainty detection.'
    },
    {
        'area': 'Off-domain Safety',
        'condition_v1': f'Jumlah error/warning Stage 07: {error_count}',
        'gap': 'Beberapa input di luar domain masih bisa memicu rekomendasi.',
        'target_v2': 'Fallback dan off-domain guardrail lebih tegas.',
        'priority': 'Medium',
        'stage10_action': 'Tambahkan negative examples dan thresholding berbasis confidence.'
    },
    {
        'area': 'Explainability',
        'condition_v1': 'Alasan rekomendasi masih dominan keyword/text similarity.',
        'gap': 'Belum menjelaskan score breakdown dan trade-off jika pilihan awal siswa kurang selaras.',
        'target_v2': 'Setiap rekomendasi menampilkan score component: RIASEC, MI, VARK, school fit, career fit, readiness, dan initial choice alignment.',
        'priority': 'High',
        'stage10_action': 'Bangun scoring config dan response blueprint.'
    },
    {
        'area': 'Production Governance',
        'condition_v1': 'Belum ada data dictionary, versioning taxonomy, dan audit trail rekomendasi.',
        'gap': 'Sulit diaudit dan divalidasi oleh domain expert.',
        'target_v2': 'Setiap taxonomy memiliki schema, version, source_note, dan validation_status.',
        'priority': 'Medium',
        'stage10_action': 'Tambahkan taxonomy metadata dan output logging.'
    },
])

gap_output = REPORT_STAGE09_DIR / 'gap_analysis_v1_to_v2.csv'
gap_analysis.to_csv(gap_output, index=False)

display(gap_analysis)
print('Saved:', gap_output)

,area,condition_v1,gap,target_v2,priority,stage10_action
0,Dataset Intent,"12 utterance, 8 intent",Belum cukup untuk model intent yang robust dan generalizable.,"10.000–15.000 synthetic utterances dengan variasi bahasa formal, non-formal, Jawa umum, Sunda umum, dan off-domain.",High,Bangun synthetic data generator dan balanced scenario sampling.
1,Taxonomy Program Studi,5 program studi S1,Cakupan prodi terlalu sempit sehingga rekomendasi bias ke beberapa prodi awal.,Minimal 30–50 prodi S1 PDDikti-style dengan rumpun/bidang ilmu dan profil kecocokan.,High,Perluas katalog prodi dan validasi nama prodi terhadap referensi PDDikti.
2,Input Profil Siswa,Field kelas masih muncul sebagai variabel utama.,Tidak merepresentasikan konteks Indonesia seperti SMA/MA/SMK/MAK dan jurusan/konsentrasi siswa.,"Gunakan school_type, school_major_group, school_major_detail, dan subject_strength.",High,Ganti field kelas menjadi taxonomy sekolah dan jurusan asal siswa.
3,Psychopedagogical Mapping,"Minat, hobi, mapel, dan gaya belajar masih berupa keyword bebas.","Belum ada standardisasi RIASEC, Multiple Intelligences, VARK, Grit/Mindset, dan Career Alignment.",Semua profil siswa dipetakan ke taxonomy eksplisit dan score component.,High,Bangun taxonomy CSV dan extractor logic.
4,Intent Routing dan Klarifikasi,Akurasi klarifikasi_minat: 0.0,Input siswa yang ragu/bingung masih dapat diarahkan langsung ke rekomendasi.,"Intent klarifikasi_minat harus memicu guided profiling, bukan langsung Top-N prodi.",High,Tambahkan guided question flow dan rule untuk uncertainty detection.
5,Off-domain Safety,Jumlah error/warning Stage 07: 4,Beberapa input di luar domain masih bisa memicu rekomendasi.,Fallback dan off-domain guardrail lebih tegas.,Medium,Tambahkan negative examples dan thresholding berbasis confidence.
6,Explainability,Alasan rekomendasi masih dominan keyword/text similarity.,Belum menjelaskan score breakdown dan trade-off jika pilihan awal siswa kurang selaras.,"Setiap rekomendasi menampilkan score component: RIASEC, MI, VARK, school fit, career fit, readiness, dan initial cho...",High,Bangun scoring config dan response blueprint.
7,Production Governance,"Belum ada data dictionary, versioning taxonomy, dan audit trail rekomendasi.",Sulit diaudit dan divalidasi oleh domain expert.,"Setiap taxonomy memiliki schema, version, source_note, dan validation_status.",Medium,Tambahkan taxonomy metadata dan output logging.


Saved: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\reports\stage09\gap_analysis_v1_to_v2.csv


## 5. Functional Requirement v2

Functional requirement menjelaskan fungsi utama yang wajib tersedia agar chatbot dapat bekerja sebagai sistem rekomendasi program studi yang lebih siap produksi.

In [4]:
# 4. Functional Requirement v2

functional_requirements = pd.DataFrame([
    ['FR-001', 'Profiling Sekolah', 'Sistem menerima input jenis sekolah siswa: SMA, MA, SMK, atau MAK.', 'Must Have', 'Input Profil'],
    ['FR-002', 'Profiling Jurusan/Konsentrasi', 'Sistem menerima jurusan atau konsentrasi siswa sesuai konteks sekolah Indonesia.', 'Must Have', 'Input Profil'],
    ['FR-003', 'Pilihan Program Awal', 'Sistem menyediakan fitur Pilih Jurusan Awal untuk membandingkan intuisi awal siswa dengan hasil pemetaan.', 'Must Have', 'Input Profil'],
    ['FR-004', 'Intent Classification', 'Sistem mengklasifikasikan intent: sapaan, rekomendasi_prodi, roadmap_belajar, prospek_karier, skill_awal, info_program_studi, klarifikasi_minat, bandingkan_pilihan_awal, fallback.', 'Must Have', 'NLP'],
    ['FR-005', 'Guided Profiling', 'Jika siswa bingung atau belum mengetahui minat, sistem menanyakan pertanyaan klarifikasi bertahap.', 'Must Have', 'Conversation Flow'],
    ['FR-006', 'RIASEC Mapping', 'Sistem memetakan minat/passion ke Realistic, Investigative, Artistic, Social, Enterprising, dan Conventional.', 'Must Have', 'Taxonomy Mapping'],
    ['FR-007', 'Multiple Intelligences Mapping', 'Sistem memetakan kecerdasan dominan ke Linguistik, Visual-Spasial, Musik, Interpersonal, Logis-Matematis, Kinestetik, Intrapersonal, dan Naturalis.', 'Must Have', 'Taxonomy Mapping'],
    ['FR-008', 'VARK Mapping', 'Sistem memetakan gaya belajar ke Visual, Reading/Writing, Auditory, dan Kinesthetic.', 'Must Have', 'Taxonomy Mapping'],
    ['FR-009', 'Grit/Mindset Mapping', 'Sistem menilai indikator ketahanan dan motivasi belajar: grit level, growth mindset, fixed mindset, mixed mindset.', 'Should Have', 'Taxonomy Mapping'],
    ['FR-010', 'Career Alignment Mapping', 'Sistem memetakan orientasi karier ke Akademik, Sosial, Praktikal, Kreatif, dan Eksekutif.', 'Must Have', 'Taxonomy Mapping'],
    ['FR-011', 'Hybrid Recommender', 'Sistem menghasilkan Top-N prodi menggunakan kombinasi text similarity, structured score, taxonomy fit, dan initial choice alignment.', 'Must Have', 'Recommender'],
    ['FR-012', 'Explanation Engine', 'Sistem menampilkan alasan rekomendasi dalam bentuk score breakdown dan narasi mudah dipahami.', 'Must Have', 'Explainability'],
    ['FR-013', 'Roadmap Generator', 'Sistem memberikan roadmap belajar 3–6 bulan awal berdasarkan prodi prioritas.', 'Should Have', 'Roadmap'],
    ['FR-014', 'Safety Disclaimer', 'Sistem menyatakan bahwa rekomendasi bersifat awal dan perlu divalidasi dengan guru BK/orang tua/konselor.', 'Must Have', 'Safety'],
    ['FR-015', 'Audit Output', 'Sistem menyimpan output rekomendasi, score component, dan taxonomy version untuk evaluasi.', 'Should Have', 'Governance'],
], columns=['requirement_id', 'requirement_name', 'description', 'priority', 'module'])

functional_output = REPORT_STAGE09_DIR / 'functional_requirements_v2.csv'
functional_requirements.to_csv(functional_output, index=False)

display(functional_requirements)
print('Saved:', functional_output)

,requirement_id,requirement_name,description,priority,module
0,FR-001,Profiling Sekolah,"Sistem menerima input jenis sekolah siswa: SMA, MA, SMK, atau MAK.",Must Have,Input Profil
1,FR-002,Profiling Jurusan/Konsentrasi,Sistem menerima jurusan atau konsentrasi siswa sesuai konteks sekolah Indonesia.,Must Have,Input Profil
2,FR-003,Pilihan Program Awal,Sistem menyediakan fitur Pilih Jurusan Awal untuk membandingkan intuisi awal siswa dengan hasil pemetaan.,Must Have,Input Profil
3,FR-004,Intent Classification,"Sistem mengklasifikasikan intent: sapaan, rekomendasi_prodi, roadmap_belajar, prospek_karier, skill_awal, info_progr...",Must Have,NLP
4,FR-005,Guided Profiling,"Jika siswa bingung atau belum mengetahui minat, sistem menanyakan pertanyaan klarifikasi bertahap.",Must Have,Conversation Flow
5,FR-006,RIASEC Mapping,"Sistem memetakan minat/passion ke Realistic, Investigative, Artistic, Social, Enterprising, dan Conventional.",Must Have,Taxonomy Mapping
6,FR-007,Multiple Intelligences Mapping,"Sistem memetakan kecerdasan dominan ke Linguistik, Visual-Spasial, Musik, Interpersonal, Logis-Matematis, Kinestetik...",Must Have,Taxonomy Mapping
7,FR-008,VARK Mapping,"Sistem memetakan gaya belajar ke Visual, Reading/Writing, Auditory, dan Kinesthetic.",Must Have,Taxonomy Mapping
8,FR-009,Grit/Mindset Mapping,"Sistem menilai indikator ketahanan dan motivasi belajar: grit level, growth mindset, fixed mindset, mixed mindset.",Should Have,Taxonomy Mapping
9,FR-010,Career Alignment Mapping,"Sistem memetakan orientasi karier ke Akademik, Sosial, Praktikal, Kreatif, dan Eksekutif.",Must Have,Taxonomy Mapping


Saved: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\reports\stage09\functional_requirements_v2.csv


## 6. Non-Functional Requirement v2

Non-functional requirement menentukan kualitas sistem yang diharapkan ketika chatbot bergerak menuju production-oriented prototype.

In [5]:
# 5. Non-Functional Requirement v2

non_functional_requirements = pd.DataFrame([
    ['NFR-001', 'Explainability', 'Setiap rekomendasi wajib memiliki score breakdown dan alasan naratif.', 'High'],
    ['NFR-002', 'Maintainability', 'Taxonomy disimpan sebagai CSV/JSON terpisah agar mudah diperbarui tanpa mengubah kode utama.', 'High'],
    ['NFR-003', 'Data Quality', 'Dataset synthetic harus memiliki label seimbang, tidak duplikatif berlebihan, dan memiliki metadata source_type.', 'High'],
    ['NFR-004', 'Scalability', 'Struktur prodi dan taxonomy harus dapat diperluas dari 30 prodi ke ratusan prodi.', 'Medium'],
    ['NFR-005', 'Robustness', 'Sistem harus tahan terhadap variasi bahasa formal, non-formal, typo ringan, Jawa umum, dan Sunda umum.', 'High'],
    ['NFR-006', 'Safety', 'Sistem tidak boleh memberi diagnosis psikologis atau memaksa siswa memilih satu jurusan.', 'High'],
    ['NFR-007', 'Privacy', 'Sistem tidak meminta data sensitif seperti NIK, alamat lengkap, data kesehatan rinci, atau nilai rapor detail jika tidak diperlukan.', 'High'],
    ['NFR-008', 'Performance', 'Respons chatbot Streamlit idealnya muncul kurang dari 3 detik untuk model klasik berbasis scikit-learn.', 'Medium'],
    ['NFR-009', 'Versioning', 'Setiap taxonomy memiliki taxonomy_version, created_at, updated_at, dan validation_status.', 'Medium'],
    ['NFR-010', 'Evaluation Readiness', 'Sistem memiliki test scenario untuk intent, routing, rekomendasi, fallback, dan initial choice comparison.', 'High'],
], columns=['requirement_id', 'quality_attribute', 'description', 'priority'])

non_functional_output = REPORT_STAGE09_DIR / 'non_functional_requirements_v2.csv'
non_functional_requirements.to_csv(non_functional_output, index=False)

display(non_functional_requirements)
print('Saved:', non_functional_output)

,requirement_id,quality_attribute,description,priority
0,NFR-001,Explainability,Setiap rekomendasi wajib memiliki score breakdown dan alasan naratif.,High
1,NFR-002,Maintainability,Taxonomy disimpan sebagai CSV/JSON terpisah agar mudah diperbarui tanpa mengubah kode utama.,High
2,NFR-003,Data Quality,"Dataset synthetic harus memiliki label seimbang, tidak duplikatif berlebihan, dan memiliki metadata source_type.",High
3,NFR-004,Scalability,Struktur prodi dan taxonomy harus dapat diperluas dari 30 prodi ke ratusan prodi.,Medium
4,NFR-005,Robustness,"Sistem harus tahan terhadap variasi bahasa formal, non-formal, typo ringan, Jawa umum, dan Sunda umum.",High
5,NFR-006,Safety,Sistem tidak boleh memberi diagnosis psikologis atau memaksa siswa memilih satu jurusan.,High
6,NFR-007,Privacy,"Sistem tidak meminta data sensitif seperti NIK, alamat lengkap, data kesehatan rinci, atau nilai rapor detail jika t...",High
7,NFR-008,Performance,Respons chatbot Streamlit idealnya muncul kurang dari 3 detik untuk model klasik berbasis scikit-learn.,Medium
8,NFR-009,Versioning,"Setiap taxonomy memiliki taxonomy_version, created_at, updated_at, dan validation_status.",Medium
9,NFR-010,Evaluation Readiness,"Sistem memiliki test scenario untuk intent, routing, rekomendasi, fallback, dan initial choice comparison.",High


Saved: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\reports\stage09\non_functional_requirements_v2.csv


## 7. Redesign User Persona v2

Persona v2 harus menggambarkan variasi siswa pendidikan menengah Indonesia secara lebih realistis. Persona ini akan menjadi dasar pembuatan skenario synthetic chat pada tahap berikutnya.

In [6]:
# 6. User Persona v2

user_personas = pd.DataFrame([
    ['PER-001', 'Siswa SMA akademik-saintek', 'SMA/MA', 'MIPA/Saintek/Fase F mapel sains', 'Tertarik matematika, biologi, fisika, kimia, data, teknologi.', 'Butuh validasi antara Kedokteran, Teknik, Data, Farmasi.', 'rekomendasi_prodi, bandingkan_pilihan_awal, roadmap_belajar'],
    ['PER-002', 'Siswa SMA sosial-humaniora', 'SMA/MA', 'IPS/Soshum/Fase F mapel sosial', 'Tertarik ekonomi, hukum, komunikasi, psikologi, kebijakan publik.', 'Bingung memilih antara Manajemen, Hukum, Psikologi, Ilmu Komunikasi.', 'rekomendasi_prodi, info_program_studi, prospek_karier'],
    ['PER-003', 'Siswa SMA bahasa-kreatif', 'SMA/MA', 'Bahasa/Seni/Fase F mapel bahasa dan seni', 'Tertarik menulis, desain, konten, bahasa asing, public speaking.', 'Butuh prodi kreatif yang prospek kariernya jelas.', 'rekomendasi_prodi, skill_awal, roadmap_belajar'],
    ['PER-004', 'Siswa SMK teknologi informasi', 'SMK/MAK', 'RPL/TKJ/SIJA/Pengembangan Gim', 'Suka coding, jaringan, aplikasi, game, troubleshooting.', 'Ingin tahu apakah lanjut TI, SI, Sains Data, atau langsung kerja.', 'bandingkan_pilihan_awal, roadmap_belajar, skill_awal'],
    ['PER-005', 'Siswa SMK bisnis-manajemen', 'SMK/MAK', 'Akuntansi/Manajemen Perkantoran/Pemasaran/Bisnis Digital', 'Suka bisnis, administrasi, pemasaran, laporan keuangan, toko online.', 'Butuh mapping ke Akuntansi, Manajemen, Bisnis Digital, Sistem Informasi.', 'rekomendasi_prodi, prospek_karier'],
    ['PER-006', 'Siswa SMK kesehatan', 'SMK/MAK', 'Layanan Kesehatan/Farmasi/Kesejahteraan Sosial', 'Suka membantu orang, praktik laboratorium, kesehatan dasar.', 'Bingung antara Keperawatan, Farmasi, Kesehatan Masyarakat, Gizi.', 'rekomendasi_prodi, bandingkan_pilihan_awal'],
    ['PER-007', 'Siswa ragu/belum mengenal minat', 'SMA/MA/SMK/MAK', 'Belum spesifik', 'Mengatakan bingung, belum tahu minat, takut salah jurusan.', 'Tidak boleh langsung diberi rekomendasi final; perlu guided profiling.', 'klarifikasi_minat, guided_profiling'],
    ['PER-008', 'Siswa dengan pilihan awal kuat tetapi belum tentu selaras', 'SMA/MA/SMK/MAK', 'Beragam', 'Sudah memilih Kedokteran/TI/Psikologi/Hukum karena tren/keluarga.', 'Perlu perbandingan antara intuisi awal dan hasil scoring profil.', 'bandingkan_pilihan_awal, rekomendasi_prodi'],
], columns=['persona_id', 'persona_name', 'school_type', 'school_major_context', 'profile_signal', 'main_problem', 'dominant_intent'])

persona_output = REPORT_STAGE09_DIR / 'user_persona_v2.csv'
user_personas.to_csv(persona_output, index=False)

display(user_personas)
print('Saved:', persona_output)

,persona_id,persona_name,school_type,school_major_context,profile_signal,main_problem,dominant_intent
0,PER-001,Siswa SMA akademik-saintek,SMA/MA,MIPA/Saintek/Fase F mapel sains,"Tertarik matematika, biologi, fisika, kimia, data, teknologi.","Butuh validasi antara Kedokteran, Teknik, Data, Farmasi.","rekomendasi_prodi, bandingkan_pilihan_awal, roadmap_belajar"
1,PER-002,Siswa SMA sosial-humaniora,SMA/MA,IPS/Soshum/Fase F mapel sosial,"Tertarik ekonomi, hukum, komunikasi, psikologi, kebijakan publik.","Bingung memilih antara Manajemen, Hukum, Psikologi, Ilmu Komunikasi.","rekomendasi_prodi, info_program_studi, prospek_karier"
2,PER-003,Siswa SMA bahasa-kreatif,SMA/MA,Bahasa/Seni/Fase F mapel bahasa dan seni,"Tertarik menulis, desain, konten, bahasa asing, public speaking.",Butuh prodi kreatif yang prospek kariernya jelas.,"rekomendasi_prodi, skill_awal, roadmap_belajar"
3,PER-004,Siswa SMK teknologi informasi,SMK/MAK,RPL/TKJ/SIJA/Pengembangan Gim,"Suka coding, jaringan, aplikasi, game, troubleshooting.","Ingin tahu apakah lanjut TI, SI, Sains Data, atau langsung kerja.","bandingkan_pilihan_awal, roadmap_belajar, skill_awal"
4,PER-005,Siswa SMK bisnis-manajemen,SMK/MAK,Akuntansi/Manajemen Perkantoran/Pemasaran/Bisnis Digital,"Suka bisnis, administrasi, pemasaran, laporan keuangan, toko online.","Butuh mapping ke Akuntansi, Manajemen, Bisnis Digital, Sistem Informasi.","rekomendasi_prodi, prospek_karier"
5,PER-006,Siswa SMK kesehatan,SMK/MAK,Layanan Kesehatan/Farmasi/Kesejahteraan Sosial,"Suka membantu orang, praktik laboratorium, kesehatan dasar.","Bingung antara Keperawatan, Farmasi, Kesehatan Masyarakat, Gizi.","rekomendasi_prodi, bandingkan_pilihan_awal"
6,PER-007,Siswa ragu/belum mengenal minat,SMA/MA/SMK/MAK,Belum spesifik,"Mengatakan bingung, belum tahu minat, takut salah jurusan.",Tidak boleh langsung diberi rekomendasi final; perlu guided profiling.,"klarifikasi_minat, guided_profiling"
7,PER-008,Siswa dengan pilihan awal kuat tetapi belum tentu selaras,SMA/MA/SMK/MAK,Beragam,Sudah memilih Kedokteran/TI/Psikologi/Hukum karena tren/keluarga.,Perlu perbandingan antara intuisi awal dan hasil scoring profil.,"bandingkan_pilihan_awal, rekomendasi_prodi"


Saved: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\reports\stage09\user_persona_v2.csv


## 8. Redesign Taxonomy Sekolah Indonesia

Pada v2, input `kelas` tidak lagi menjadi opsi utama. Sistem menggunakan konteks pendidikan menengah Indonesia:

- **School type:** SMA, MA, SMK, MAK.
- **School stream/group:** umum, madrasah umum, vokasi, madrasah vokasi.
- **Major/detail:** jurusan lama seperti IPA/IPS/Bahasa/Keagamaan, kelompok mata pelajaran pilihan Kurikulum Merdeka, atau konsentrasi keahlian SMK/MAK.

Catatan: taxonomy SMK/MAK pada cell ini adalah subset awal yang cukup untuk prototype v2. Untuk production penuh, daftar konsentrasi perlu dilengkapi dari sumber resmi spektrum keahlian SMK/MAK.

In [7]:
# 7. Taxonomy Sekolah Indonesia v2

school_taxonomy = pd.DataFrame([
    # SMA/MA umum dan Kurikulum Merdeka/Fase F
    ['SCH-SMA-001', 'SMA', 'Umum', 'MIPA/IPA', 'Matematika, Fisika, Kimia, Biologi', 'Sains, Teknik, Kesehatan, Komputasi', 'Matematika|Fisika|Kimia|Biologi|Informatika', 'SMA jalur lama atau sekolah yang masih memakai peminatan'],
    ['SCH-SMA-002', 'SMA', 'Umum', 'IPS/Soshum', 'Ekonomi, Sosiologi, Geografi, Sejarah', 'Bisnis, Hukum, Komunikasi, Psikologi, Administrasi', 'Ekonomi|Sosiologi|Geografi|Sejarah', 'SMA jalur lama atau preferensi mapel sosial-humaniora'],
    ['SCH-SMA-003', 'SMA', 'Umum', 'Bahasa dan Budaya', 'Bahasa Indonesia, Bahasa Inggris, Bahasa asing, Sastra', 'Bahasa, Komunikasi, Sastra, Hubungan Internasional', 'Bahasa Indonesia|Bahasa Inggris|Sastra|Public Speaking', 'SMA jalur lama atau preferensi bahasa'],
    ['SCH-SMA-004', 'SMA', 'Umum', 'Seni/Kreatif', 'Seni Budaya, Desain, Musik, Konten Kreatif', 'DKV, Desain Produk, Seni, Film, Musik', 'Seni|Desain|Musik|Kreatif|Konten', 'Kelompok minat kreatif'],
    ['SCH-SMA-005', 'SMA', 'Umum', 'Pilihan Campuran Kurikulum Merdeka', 'Kombinasi mapel sains, sosial, bahasa, atau seni', 'Lintas rumpun sesuai kombinasi mapel', 'Mapel Pilihan|Fase F|Campuran', 'Untuk siswa kelas XI/XII yang memilih mapel fleksibel'],
    ['SCH-MA-001', 'MA', 'Madrasah Umum', 'MIPA/IPA', 'Matematika, Fisika, Kimia, Biologi, keagamaan', 'Sains, Kesehatan, Teknik, Pendidikan', 'Matematika|Biologi|Kimia|Fisika|Keagamaan', 'Madrasah Aliyah dengan peminatan sains'],
    ['SCH-MA-002', 'MA', 'Madrasah Umum', 'IPS/Soshum', 'Ekonomi, Sosiologi, Sejarah, Keagamaan', 'Sosial, Hukum, Ekonomi, Pendidikan, Dakwah', 'Ekonomi|Sosiologi|Sejarah|Keagamaan', 'Madrasah Aliyah dengan peminatan sosial'],
    ['SCH-MA-003', 'MA', 'Madrasah Umum', 'Keagamaan', 'Ilmu agama, bahasa Arab, studi Islam', 'Pendidikan Agama, Hukum Keluarga, Komunikasi Penyiaran Islam', 'Agama|Bahasa Arab|Pendidikan|Sosial', 'Untuk MA program keagamaan'],
    # SMK/MAK Teknologi Informasi
    ['SCH-SMK-001', 'SMK', 'Vokasi', 'Pengembangan Perangkat Lunak dan Gim - RPL', 'Pemrograman, basis data, aplikasi, web, mobile', 'Teknik Informatika, Sistem Informasi, Sains Data, RPL, Bisnis Digital', 'Coding|RPL|Web|Mobile|Database|Algoritma', 'Subset spektrum keahlian SMK/MAK'],
    ['SCH-SMK-002', 'SMK', 'Vokasi', 'Teknik Jaringan Komputer dan Telekomunikasi - TKJ', 'Jaringan, server, troubleshooting, cloud dasar', 'Teknik Informatika, Teknologi Informasi, Sistem Informasi, Teknik Telekomunikasi', 'Jaringan|Server|Komputer|Cloud|Cybersecurity', 'Subset spektrum keahlian SMK/MAK'],
    ['SCH-SMK-003', 'SMK', 'Vokasi', 'Sistem Informasi, Jaringan, dan Aplikasi - SIJA', 'Aplikasi, jaringan, sistem informasi, integrasi teknologi', 'Sistem Informasi, Teknik Informatika, Sains Data', 'Sistem Informasi|Jaringan|Aplikasi|Database', 'Subset spektrum keahlian SMK/MAK'],
    ['SCH-SMK-004', 'SMK', 'Vokasi', 'Pengembangan Gim', 'Game design, game programming, animasi dasar', 'Teknik Informatika, Desain Komunikasi Visual, Animasi', 'Game|Gim|Unity|Desain Game|Animasi', 'Subset spektrum keahlian SMK/MAK'],
    # SMK/MAK Bisnis
    ['SCH-SMK-005', 'SMK', 'Vokasi', 'Akuntansi dan Keuangan Lembaga', 'Pembukuan, laporan keuangan, pajak, spreadsheet', 'Akuntansi, Manajemen, Sistem Informasi, Perpajakan', 'Akuntansi|Keuangan|Pajak|Excel|Administrasi', 'Konsentrasi vokasi bisnis/keuangan'],
    ['SCH-SMK-006', 'SMK', 'Vokasi', 'Manajemen Perkantoran dan Layanan Bisnis', 'Administrasi, dokumen, layanan bisnis, komunikasi', 'Manajemen, Administrasi Bisnis, Sistem Informasi, Ilmu Komunikasi', 'Administrasi|Perkantoran|Dokumen|Komunikasi', 'Konsentrasi vokasi bisnis/administrasi'],
    ['SCH-SMK-007', 'SMK', 'Vokasi', 'Pemasaran - Bisnis Digital', 'Digital marketing, e-commerce, promosi, customer service', 'Bisnis Digital, Manajemen, Ilmu Komunikasi, Sistem Informasi', 'Marketing|Bisnis Digital|Ecommerce|Konten|Retail', 'Konsentrasi vokasi bisnis digital'],
    # SMK/MAK Kesehatan, Kreatif, Teknik, Agribisnis
    ['SCH-SMK-008', 'SMK', 'Vokasi', 'Layanan Kesehatan', 'Pelayanan kesehatan dasar, caregiving, kesehatan masyarakat', 'Keperawatan, Kesehatan Masyarakat, Ilmu Gizi, Farmasi', 'Kesehatan|Caregiving|Keperawatan|Biologi', 'Konsentrasi vokasi kesehatan'],
    ['SCH-SMK-009', 'SMK', 'Vokasi', 'Farmasi Klinis dan Komunitas', 'Obat, pelayanan farmasi, kimia dasar, kesehatan', 'Farmasi, Kimia, Kesehatan Masyarakat', 'Farmasi|Obat|Kimia|Kesehatan', 'Konsentrasi vokasi kesehatan/farmasi'],
    ['SCH-SMK-010', 'SMK', 'Vokasi', 'Desain Komunikasi Visual', 'Desain grafis, ilustrasi, branding, konten visual', 'DKV, Desain Produk, Ilmu Komunikasi, Animasi', 'Desain|Visual|Ilustrasi|Branding|Konten', 'Konsentrasi vokasi kreatif'],
    ['SCH-SMK-011', 'SMK', 'Vokasi', 'Teknik Mesin', 'Produksi, manufaktur, perawatan mesin, gambar teknik', 'Teknik Mesin, Teknik Industri, Pendidikan Teknik Mesin', 'Mesin|Manufaktur|CAD|Perawatan|Produksi', 'Konsentrasi vokasi teknik'],
    ['SCH-SMK-012', 'SMK', 'Vokasi', 'Teknik Elektronika', 'Elektronika, sensor, kontrol, perangkat keras', 'Teknik Elektro, Teknik Komputer, Teknik Mekatronika', 'Elektronika|Sensor|Robotik|Kontrol|IoT', 'Konsentrasi vokasi teknik'],
    ['SCH-SMK-013', 'SMK', 'Vokasi', 'Agribisnis Tanaman/Ternak/Perikanan', 'Budidaya, agribisnis, lingkungan, produksi pangan', 'Agribisnis, Agroteknologi, Peternakan, Perikanan, Teknologi Pangan', 'Pertanian|Agribisnis|Ternak|Perikanan|Pangan', 'Konsentrasi vokasi agribisnis'],
    ['SCH-MAK-001', 'MAK', 'Madrasah Vokasi', 'MAK Teknologi/Bisnis/Kesehatan/Kreatif', 'Konsentrasi kejuruan berbasis madrasah aliyah kejuruan', 'Mengikuti mapping SMK/MAK sesuai konsentrasi', 'MAK|Vokasi|Keagamaan|Konsentrasi', 'Gunakan detail konsentrasi seperti SMK/MAK'],
], columns=['school_taxonomy_id', 'school_type', 'school_group', 'school_major_detail', 'subject_context', 'recommended_higher_education_area', 'keywords', 'notes'])

school_output = TAXONOMY_DIR / 'school_taxonomy_v2.csv'
school_taxonomy.to_csv(school_output, index=False)

display(school_taxonomy)
print('Saved:', school_output)

,school_taxonomy_id,school_type,school_group,school_major_detail,subject_context,recommended_higher_education_area,keywords,notes
0,SCH-SMA-001,SMA,Umum,MIPA/IPA,"Matematika, Fisika, Kimia, Biologi","Sains, Teknik, Kesehatan, Komputasi",Matematika|Fisika|Kimia|Biologi|Informatika,SMA jalur lama atau sekolah yang masih memakai peminatan
1,SCH-SMA-002,SMA,Umum,IPS/Soshum,"Ekonomi, Sosiologi, Geografi, Sejarah","Bisnis, Hukum, Komunikasi, Psikologi, Administrasi",Ekonomi|Sosiologi|Geografi|Sejarah,SMA jalur lama atau preferensi mapel sosial-humaniora
2,SCH-SMA-003,SMA,Umum,Bahasa dan Budaya,"Bahasa Indonesia, Bahasa Inggris, Bahasa asing, Sastra","Bahasa, Komunikasi, Sastra, Hubungan Internasional",Bahasa Indonesia|Bahasa Inggris|Sastra|Public Speaking,SMA jalur lama atau preferensi bahasa
3,SCH-SMA-004,SMA,Umum,Seni/Kreatif,"Seni Budaya, Desain, Musik, Konten Kreatif","DKV, Desain Produk, Seni, Film, Musik",Seni|Desain|Musik|Kreatif|Konten,Kelompok minat kreatif
4,SCH-SMA-005,SMA,Umum,Pilihan Campuran Kurikulum Merdeka,"Kombinasi mapel sains, sosial, bahasa, atau seni",Lintas rumpun sesuai kombinasi mapel,Mapel Pilihan|Fase F|Campuran,Untuk siswa kelas XI/XII yang memilih mapel fleksibel
5,SCH-MA-001,MA,Madrasah Umum,MIPA/IPA,"Matematika, Fisika, Kimia, Biologi, keagamaan","Sains, Kesehatan, Teknik, Pendidikan",Matematika|Biologi|Kimia|Fisika|Keagamaan,Madrasah Aliyah dengan peminatan sains
6,SCH-MA-002,MA,Madrasah Umum,IPS/Soshum,"Ekonomi, Sosiologi, Sejarah, Keagamaan","Sosial, Hukum, Ekonomi, Pendidikan, Dakwah",Ekonomi|Sosiologi|Sejarah|Keagamaan,Madrasah Aliyah dengan peminatan sosial
7,SCH-MA-003,MA,Madrasah Umum,Keagamaan,"Ilmu agama, bahasa Arab, studi Islam","Pendidikan Agama, Hukum Keluarga, Komunikasi Penyiaran Islam",Agama|Bahasa Arab|Pendidikan|Sosial,Untuk MA program keagamaan
8,SCH-SMK-001,SMK,Vokasi,Pengembangan Perangkat Lunak dan Gim - RPL,"Pemrograman, basis data, aplikasi, web, mobile","Teknik Informatika, Sistem Informasi, Sains Data, RPL, Bisnis Digital",Coding|RPL|Web|Mobile|Database|Algoritma,Subset spektrum keahlian SMK/MAK
9,SCH-SMK-002,SMK,Vokasi,Teknik Jaringan Komputer dan Telekomunikasi - TKJ,"Jaringan, server, troubleshooting, cloud dasar","Teknik Informatika, Teknologi Informasi, Sistem Informasi, Teknik Telekomunikasi",Jaringan|Server|Komputer|Cloud|Cybersecurity,Subset spektrum keahlian SMK/MAK


Saved: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\data\taxonomy\school_taxonomy_v2.csv


## 9. Redesign Taxonomy Program Studi Indonesia / PDDikti-style

Tujuan taxonomy ini adalah menyediakan katalog program studi S1 yang lebih luas dan terstruktur. Field `pddikti_reference_status` digunakan agar desain tetap jujur secara akademik:

- `prototype_pddikti_style`: nama prodi disusun mengikuti gaya umum penamaan prodi Indonesia dan perlu divalidasi lagi ke PDDikti.
- `validated_pddikti`: dapat digunakan nanti setelah nama dan kode prodi benar-benar diverifikasi terhadap PDDikti.

Pada tahap ini **kode PDDikti tidak diisi secara spekulatif**. Jika dibutuhkan pada tahap berikutnya, kode prodi harus diambil dari sumber resmi atau dataset referensi yang valid.

In [8]:
# 8. Taxonomy Program Studi S1 Indonesia/PDDikti-style v2

program_taxonomy_rows = [
    # Komputer dan Informatika
    ['PRG2-001', 'Teknik Informatika', 'S1', 'Komputer/Informatika', 'Teknologi', 'Investigative', 'Realistic', 'Logis-Matematis', 'Kinesthetic', 'Praktikal', 'tinggi', 'tinggi', 'sedang', 'rendah', 'SMA MIPA; SMK RPL/TKJ/SIJA', 'Software Engineer; Backend Developer; AI Engineer', 'coding|algoritma|software|komputer|aplikasi', 'prototype_pddikti_style'],
    ['PRG2-002', 'Sistem Informasi', 'S1', 'Komputer/Informatika', 'Teknologi-Bisnis', 'Conventional', 'Enterprising', 'Logis-Matematis', 'Reading/Writing', 'Eksekutif', 'sedang', 'sedang', 'tinggi', 'rendah', 'SMA IPS/MIPA; SMK RPL/TKJ/Bisnis', 'Business Analyst; System Analyst; ERP Consultant', 'sistem informasi|bisnis|analisis sistem|erp|proses bisnis', 'prototype_pddikti_style'],
    ['PRG2-003', 'Sains Data', 'S1', 'Komputer/Informatika', 'Data dan AI', 'Investigative', 'Conventional', 'Logis-Matematis', 'Visual', 'Akademik', 'sangat tinggi', 'tinggi', 'sedang', 'rendah', 'SMA MIPA; SMK RPL/SIJA', 'Data Analyst; Data Scientist; ML Engineer', 'data science|statistika|machine learning|dashboard|analisis data', 'prototype_pddikti_style'],
    ['PRG2-004', 'Teknologi Informasi', 'S1', 'Komputer/Informatika', 'Infrastruktur dan Sistem', 'Realistic', 'Investigative', 'Logis-Matematis', 'Kinesthetic', 'Praktikal', 'tinggi', 'tinggi', 'sedang', 'rendah', 'SMA MIPA; SMK TKJ/SIJA', 'IT Infrastructure Engineer; Cloud Engineer; Network Engineer', 'jaringan|server|cloud|infrastruktur|cybersecurity', 'prototype_pddikti_style'],
    ['PRG2-005', 'Bisnis Digital', 'S1', 'Bisnis dan Manajemen', 'Teknologi-Bisnis', 'Enterprising', 'Conventional', 'Interpersonal', 'Visual', 'Eksekutif', 'sedang', 'sedang', 'tinggi', 'sedang', 'SMA IPS; SMK Bisnis Digital/Pemasaran', 'Digital Business Analyst; Product Associate; Digital Marketer', 'bisnis digital|ecommerce|startup|produk digital|marketing', 'prototype_pddikti_style'],
    # Teknik
    ['PRG2-006', 'Teknik Elektro', 'S1', 'Teknik', 'Rekayasa', 'Realistic', 'Investigative', 'Logis-Matematis', 'Kinesthetic', 'Praktikal', 'tinggi', 'sedang', 'sedang', 'rendah', 'SMA MIPA; SMK Elektronika/Ketenagalistrikan', 'Electrical Engineer; IoT Engineer; Automation Engineer', 'elektro|listrik|elektronika|sensor|iot', 'prototype_pddikti_style'],
    ['PRG2-007', 'Teknik Industri', 'S1', 'Teknik', 'Rekayasa dan Manajemen Operasi', 'Conventional', 'Enterprising', 'Logis-Matematis', 'Reading/Writing', 'Eksekutif', 'tinggi', 'sedang', 'tinggi', 'rendah', 'SMA MIPA/IPS; SMK Teknik/Bisnis', 'Industrial Engineer; Supply Chain Analyst; Quality Engineer', 'proses|produksi|manufaktur|quality|supply chain', 'prototype_pddikti_style'],
    ['PRG2-008', 'Teknik Mesin', 'S1', 'Teknik', 'Rekayasa Mekanik', 'Realistic', 'Investigative', 'Logis-Matematis', 'Kinesthetic', 'Praktikal', 'tinggi', 'rendah', 'sedang', 'rendah', 'SMA MIPA; SMK Teknik Mesin', 'Mechanical Engineer; Maintenance Engineer; Manufacturing Engineer', 'mesin|manufaktur|cad|mekanik|produksi', 'prototype_pddikti_style'],
    ['PRG2-009', 'Teknik Sipil', 'S1', 'Teknik', 'Konstruksi dan Infrastruktur', 'Realistic', 'Investigative', 'Visual-Spasial', 'Visual', 'Praktikal', 'tinggi', 'rendah', 'sedang', 'sedang', 'SMA MIPA; SMK Konstruksi', 'Civil Engineer; Construction Engineer; Quantity Surveyor', 'bangunan|konstruksi|struktur|infrastruktur|jalan', 'prototype_pddikti_style'],
    ['PRG2-010', 'Arsitektur', 'S1', 'Teknik/Desain', 'Rancang Bangun', 'Artistic', 'Realistic', 'Visual-Spasial', 'Visual', 'Kreatif', 'sedang', 'rendah', 'sedang', 'tinggi', 'SMA MIPA/Seni; SMK DPIB/DKV', 'Architect; Interior Designer; Urban Designer', 'arsitektur|desain bangunan|ruang|sketsa|visual', 'prototype_pddikti_style'],
    # Sains dasar
    ['PRG2-011', 'Matematika', 'S1', 'Sains', 'Sains Formal', 'Investigative', 'Conventional', 'Logis-Matematis', 'Reading/Writing', 'Akademik', 'sangat tinggi', 'sedang', 'sedang', 'rendah', 'SMA MIPA', 'Mathematician; Quant Analyst; Research Assistant', 'matematika|logika|teori|pemodelan|analisis', 'prototype_pddikti_style'],
    ['PRG2-012', 'Statistika', 'S1', 'Sains', 'Statistika dan Analitik', 'Investigative', 'Conventional', 'Logis-Matematis', 'Visual', 'Akademik', 'sangat tinggi', 'sedang', 'sedang', 'rendah', 'SMA MIPA; SMK RPL/Akuntansi', 'Statistician; Data Analyst; Survey Researcher', 'statistika|probabilitas|data|riset|survei', 'prototype_pddikti_style'],
    ['PRG2-013', 'Biologi', 'S1', 'Sains', 'Hayati', 'Investigative', 'Realistic', 'Naturalis', 'Reading/Writing', 'Akademik', 'sedang', 'rendah', 'sedang', 'rendah', 'SMA MIPA; SMK Kesehatan/Agribisnis', 'Biologist; Lab Analyst; Research Assistant', 'biologi|makhluk hidup|laboratorium|lingkungan|genetika', 'prototype_pddikti_style'],
    ['PRG2-014', 'Kimia', 'S1', 'Sains', 'Kimia dan Material', 'Investigative', 'Realistic', 'Logis-Matematis', 'Kinesthetic', 'Akademik', 'tinggi', 'rendah', 'sedang', 'rendah', 'SMA MIPA; SMK Kimia/Farmasi', 'Chemist; Lab Analyst; Quality Control Analyst', 'kimia|laboratorium|reaksi|analisis|material', 'prototype_pddikti_style'],
    # Kesehatan
    ['PRG2-015', 'Pendidikan Dokter', 'S1', 'Kesehatan', 'Kedokteran', 'Investigative', 'Social', 'Logis-Matematis', 'Reading/Writing', 'Akademik', 'tinggi', 'rendah', 'tinggi', 'rendah', 'SMA MIPA; kuat biologi kimia', 'Dokter; Clinical Researcher; Medical Educator', 'kedokteran|dokter|pasien|biologi|kimia|anatomi', 'prototype_pddikti_style'],
    ['PRG2-016', 'Farmasi', 'S1', 'Kesehatan', 'Farmasi dan Obat', 'Investigative', 'Conventional', 'Logis-Matematis', 'Reading/Writing', 'Akademik', 'tinggi', 'rendah', 'sedang', 'rendah', 'SMA MIPA; SMK Farmasi/Kesehatan', 'Pharmacist Assistant Pathway; QA Analyst; Regulatory Affairs', 'farmasi|obat|kimia|apoteker|laboratorium', 'prototype_pddikti_style'],
    ['PRG2-017', 'Keperawatan', 'S1', 'Kesehatan', 'Keperawatan', 'Social', 'Realistic', 'Interpersonal', 'Kinesthetic', 'Sosial', 'sedang', 'rendah', 'tinggi', 'rendah', 'SMA MIPA; SMK Kesehatan', 'Nurse; Healthcare Educator; Care Coordinator', 'keperawatan|pasien|caregiving|kesehatan|empati', 'prototype_pddikti_style'],
    ['PRG2-018', 'Kesehatan Masyarakat', 'S1', 'Kesehatan', 'Public Health', 'Social', 'Investigative', 'Interpersonal', 'Reading/Writing', 'Sosial', 'sedang', 'rendah', 'tinggi', 'rendah', 'SMA MIPA/IPS; SMK Kesehatan', 'Public Health Officer; Health Educator; Epidemiology Assistant', 'kesehatan masyarakat|epidemiologi|promosi kesehatan|komunitas', 'prototype_pddikti_style'],
    ['PRG2-019', 'Ilmu Gizi', 'S1', 'Kesehatan', 'Gizi dan Pangan', 'Investigative', 'Social', 'Naturalis', 'Reading/Writing', 'Sosial', 'sedang', 'rendah', 'tinggi', 'rendah', 'SMA MIPA; SMK Kesehatan/Tata Boga', 'Nutritionist Assistant Pathway; Food Analyst; Health Educator', 'gizi|makanan|diet|kesehatan|pangan', 'prototype_pddikti_style'],
    # Sosial humaniora dan pendidikan
    ['PRG2-020', 'Psikologi', 'S1', 'Sosial Humaniora', 'Perilaku Manusia', 'Social', 'Investigative', 'Interpersonal', 'Reading/Writing', 'Sosial', 'sedang', 'rendah', 'sangat tinggi', 'rendah', 'SMA IPS/MIPA/Bahasa', 'HR Analyst; Counselor Assistant; Research Assistant', 'psikologi|manusia|perilaku|konseling|observasi', 'prototype_pddikti_style'],
    ['PRG2-021', 'Ilmu Komunikasi', 'S1', 'Sosial Humaniora', 'Komunikasi dan Media', 'Enterprising', 'Artistic', 'Linguistik', 'Auditory', 'Kreatif', 'rendah', 'rendah', 'sangat tinggi', 'sedang', 'SMA IPS/Bahasa/Seni; SMK DKV/Pemasaran', 'PR Officer; Content Strategist; Communication Specialist', 'komunikasi|media|public speaking|konten|jurnalistik', 'prototype_pddikti_style'],
    ['PRG2-022', 'Ilmu Hukum', 'S1', 'Sosial Humaniora', 'Hukum', 'Conventional', 'Enterprising', 'Linguistik', 'Reading/Writing', 'Eksekutif', 'sedang', 'rendah', 'tinggi', 'rendah', 'SMA IPS/Bahasa/Keagamaan', 'Legal Officer; Paralegal; Compliance Staff', 'hukum|aturan|argumentasi|kontrak|keadilan', 'prototype_pddikti_style'],
    ['PRG2-023', 'Administrasi Publik', 'S1', 'Sosial Humaniora', 'Kebijakan dan Administrasi', 'Social', 'Conventional', 'Interpersonal', 'Reading/Writing', 'Eksekutif', 'sedang', 'rendah', 'tinggi', 'rendah', 'SMA IPS; MA Keagamaan/Sosial', 'Policy Analyst; Public Administrator; Program Officer', 'administrasi publik|kebijakan|pemerintahan|layanan publik', 'prototype_pddikti_style'],
    ['PRG2-024', 'Pendidikan Guru Sekolah Dasar', 'S1', 'Pendidikan', 'Pendidikan Dasar', 'Social', 'Artistic', 'Interpersonal', 'Auditory', 'Sosial', 'sedang', 'rendah', 'tinggi', 'sedang', 'SMA/MA/SMK semua jurusan dengan minat mengajar', 'Elementary Teacher; Education Content Developer', 'pgsd|mengajar|anak|pendidikan|kelas', 'prototype_pddikti_style'],
    ['PRG2-025', 'Pendidikan Bahasa Inggris', 'S1', 'Pendidikan', 'Pendidikan Bahasa', 'Social', 'Artistic', 'Linguistik', 'Auditory', 'Sosial', 'rendah', 'rendah', 'tinggi', 'sedang', 'SMA Bahasa/IPS; MA; SMK Pariwisata/Perhotelan', 'English Teacher; Translator; Education Content Creator', 'bahasa inggris|mengajar|grammar|speaking|translation', 'prototype_pddikti_style'],
    # Bisnis dan ekonomi
    ['PRG2-026', 'Manajemen', 'S1', 'Bisnis dan Manajemen', 'Manajemen Organisasi', 'Enterprising', 'Conventional', 'Interpersonal', 'Reading/Writing', 'Eksekutif', 'sedang', 'rendah', 'tinggi', 'rendah', 'SMA IPS/MIPA; SMK Bisnis', 'Management Trainee; HR Staff; Operations Staff', 'manajemen|organisasi|bisnis|leadership|operasional', 'prototype_pddikti_style'],
    ['PRG2-027', 'Akuntansi', 'S1', 'Bisnis dan Manajemen', 'Akuntansi dan Keuangan', 'Conventional', 'Investigative', 'Logis-Matematis', 'Reading/Writing', 'Eksekutif', 'tinggi', 'rendah', 'sedang', 'rendah', 'SMA IPS/MIPA; SMK Akuntansi', 'Accountant; Auditor; Tax Staff', 'akuntansi|keuangan|audit|pajak|laporan keuangan', 'prototype_pddikti_style'],
    ['PRG2-028', 'Ekonomi Pembangunan', 'S1', 'Ekonomi', 'Ekonomi dan Kebijakan', 'Investigative', 'Conventional', 'Logis-Matematis', 'Reading/Writing', 'Akademik', 'tinggi', 'rendah', 'sedang', 'rendah', 'SMA IPS/MIPA', 'Economic Analyst; Research Assistant; Policy Analyst', 'ekonomi|pembangunan|data ekonomi|kebijakan|riset', 'prototype_pddikti_style'],
    # Kreatif dan desain
    ['PRG2-029', 'Desain Komunikasi Visual', 'S1', 'Seni dan Desain', 'Desain Visual', 'Artistic', 'Enterprising', 'Visual-Spasial', 'Visual', 'Kreatif', 'rendah', 'rendah', 'sedang', 'sangat tinggi', 'SMA Seni/Bahasa; SMK DKV/Multimedia', 'Graphic Designer; UI Designer; Brand Designer', 'dkv|desain grafis|visual|ilustrasi|branding', 'prototype_pddikti_style'],
    ['PRG2-030', 'Desain Produk', 'S1', 'Seni dan Desain', 'Desain Produk', 'Artistic', 'Realistic', 'Visual-Spasial', 'Kinesthetic', 'Kreatif', 'sedang', 'rendah', 'sedang', 'sangat tinggi', 'SMA Seni/MIPA; SMK Teknik/DKV', 'Product Designer; Industrial Designer; Creative Entrepreneur', 'desain produk|produk|prototipe|visual|kreatif', 'prototype_pddikti_style'],
    ['PRG2-031', 'Film dan Televisi', 'S1', 'Seni dan Media', 'Media Kreatif', 'Artistic', 'Enterprising', 'Visual-Spasial', 'Visual', 'Kreatif', 'rendah', 'rendah', 'tinggi', 'tinggi', 'SMA Seni/Bahasa; SMK Broadcasting/Multimedia', 'Filmmaker; Video Producer; Creative Director Assistant', 'film|video|storytelling|kamera|editing', 'prototype_pddikti_style'],
    # Pertanian, pangan, pariwisata
    ['PRG2-032', 'Agribisnis', 'S1', 'Pertanian', 'Bisnis Pertanian', 'Enterprising', 'Realistic', 'Naturalis', 'Kinesthetic', 'Praktikal', 'sedang', 'rendah', 'tinggi', 'rendah', 'SMA MIPA/IPS; SMK Agribisnis', 'Agribusiness Analyst; Farm Entrepreneur; Supply Chain Staff', 'agribisnis|pertanian|usaha tani|pangan|supply chain', 'prototype_pddikti_style'],
    ['PRG2-033', 'Agroteknologi', 'S1', 'Pertanian', 'Teknologi Pertanian', 'Realistic', 'Investigative', 'Naturalis', 'Kinesthetic', 'Praktikal', 'sedang', 'rendah', 'sedang', 'rendah', 'SMA MIPA; SMK Agribisnis Tanaman', 'Agronomist; Field Officer; Research Assistant', 'agroteknologi|tanaman|budidaya|tanah|pertanian', 'prototype_pddikti_style'],
    ['PRG2-034', 'Teknologi Pangan', 'S1', 'Pangan', 'Teknologi dan Mutu Pangan', 'Investigative', 'Realistic', 'Naturalis', 'Kinesthetic', 'Praktikal', 'tinggi', 'rendah', 'sedang', 'rendah', 'SMA MIPA; SMK Tata Boga/Kimia/Pangan', 'Food Technologist; QA Food Analyst; Product Development Staff', 'teknologi pangan|makanan|mutu|laboratorium|produk pangan', 'prototype_pddikti_style'],
    ['PRG2-035', 'Pariwisata', 'S1', 'Pariwisata', 'Hospitality dan Destinasi', 'Enterprising', 'Social', 'Interpersonal', 'Auditory', 'Sosial', 'rendah', 'rendah', 'sangat tinggi', 'sedang', 'SMA IPS/Bahasa; SMK Perhotelan/Pariwisata', 'Tourism Planner; Hospitality Staff; Event Officer', 'pariwisata|hotel|hospitality|event|destinasi', 'prototype_pddikti_style'],
]

program_taxonomy = pd.DataFrame(program_taxonomy_rows, columns=[
    'program_id', 'program_name', 'jenjang', 'rumpun_ilmu', 'bidang_keilmuan',
    'riasec_primary', 'riasec_secondary', 'mi_primary', 'vark_preferred', 'career_alignment',
    'math_level', 'coding_level', 'communication_level', 'design_level',
    'school_fit', 'career_examples', 'keywords', 'pddikti_reference_status'
])

program_output = TAXONOMY_DIR / 'program_studi_taxonomy_pddikti_style_v2.csv'
program_taxonomy.to_csv(program_output, index=False)

display(program_taxonomy.head(15))
print('Total program studi v2:', len(program_taxonomy))
print('Saved:', program_output)

,program_id,program_name,jenjang,rumpun_ilmu,bidang_keilmuan,riasec_primary,riasec_secondary,mi_primary,vark_preferred,career_alignment,math_level,coding_level,communication_level,design_level,school_fit,career_examples,keywords,pddikti_reference_status
0,PRG2-001,Teknik Informatika,S1,Komputer/Informatika,Teknologi,Investigative,Realistic,Logis-Matematis,Kinesthetic,Praktikal,tinggi,tinggi,sedang,rendah,SMA MIPA; SMK RPL/TKJ/SIJA,Software Engineer; Backend Developer; AI Engineer,coding|algoritma|software|komputer|aplikasi,prototype_pddikti_style
1,PRG2-002,Sistem Informasi,S1,Komputer/Informatika,Teknologi-Bisnis,Conventional,Enterprising,Logis-Matematis,Reading/Writing,Eksekutif,sedang,sedang,tinggi,rendah,SMA IPS/MIPA; SMK RPL/TKJ/Bisnis,Business Analyst; System Analyst; ERP Consultant,sistem informasi|bisnis|analisis sistem|erp|proses bisnis,prototype_pddikti_style
2,PRG2-003,Sains Data,S1,Komputer/Informatika,Data dan AI,Investigative,Conventional,Logis-Matematis,Visual,Akademik,sangat tinggi,tinggi,sedang,rendah,SMA MIPA; SMK RPL/SIJA,Data Analyst; Data Scientist; ML Engineer,data science|statistika|machine learning|dashboard|analisis data,prototype_pddikti_style
3,PRG2-004,Teknologi Informasi,S1,Komputer/Informatika,Infrastruktur dan Sistem,Realistic,Investigative,Logis-Matematis,Kinesthetic,Praktikal,tinggi,tinggi,sedang,rendah,SMA MIPA; SMK TKJ/SIJA,IT Infrastructure Engineer; Cloud Engineer; Network Engineer,jaringan|server|cloud|infrastruktur|cybersecurity,prototype_pddikti_style
4,PRG2-005,Bisnis Digital,S1,Bisnis dan Manajemen,Teknologi-Bisnis,Enterprising,Conventional,Interpersonal,Visual,Eksekutif,sedang,sedang,tinggi,sedang,SMA IPS; SMK Bisnis Digital/Pemasaran,Digital Business Analyst; Product Associate; Digital Marketer,bisnis digital|ecommerce|startup|produk digital|marketing,prototype_pddikti_style
5,PRG2-006,Teknik Elektro,S1,Teknik,Rekayasa,Realistic,Investigative,Logis-Matematis,Kinesthetic,Praktikal,tinggi,sedang,sedang,rendah,SMA MIPA; SMK Elektronika/Ketenagalistrikan,Electrical Engineer; IoT Engineer; Automation Engineer,elektro|listrik|elektronika|sensor|iot,prototype_pddikti_style
6,PRG2-007,Teknik Industri,S1,Teknik,Rekayasa dan Manajemen Operasi,Conventional,Enterprising,Logis-Matematis,Reading/Writing,Eksekutif,tinggi,sedang,tinggi,rendah,SMA MIPA/IPS; SMK Teknik/Bisnis,Industrial Engineer; Supply Chain Analyst; Quality Engineer,proses|produksi|manufaktur|quality|supply chain,prototype_pddikti_style
7,PRG2-008,Teknik Mesin,S1,Teknik,Rekayasa Mekanik,Realistic,Investigative,Logis-Matematis,Kinesthetic,Praktikal,tinggi,rendah,sedang,rendah,SMA MIPA; SMK Teknik Mesin,Mechanical Engineer; Maintenance Engineer; Manufacturing Engineer,mesin|manufaktur|cad|mekanik|produksi,prototype_pddikti_style
8,PRG2-009,Teknik Sipil,S1,Teknik,Konstruksi dan Infrastruktur,Realistic,Investigative,Visual-Spasial,Visual,Praktikal,tinggi,rendah,sedang,sedang,SMA MIPA; SMK Konstruksi,Civil Engineer; Construction Engineer; Quantity Surveyor,bangunan|konstruksi|struktur|infrastruktur|jalan,prototype_pddikti_style
9,PRG2-010,Arsitektur,S1,Teknik/Desain,Rancang Bangun,Artistic,Realistic,Visual-Spasial,Visual,Kreatif,sedang,rendah,sedang,tinggi,SMA MIPA/Seni; SMK DPIB/DKV,Architect; Interior Designer; Urban Designer,arsitektur|desain bangunan|ruang|sketsa|visual,prototype_pddikti_style


Total program studi v2: 35
Saved: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\data\taxonomy\program_studi_taxonomy_pddikti_style_v2.csv


## 10. Taxonomy RIASEC, Multiple Intelligences, VARK, Grit/Mindset, dan Career Alignment

Taxonomy berikut digunakan untuk mengubah sinyal teks siswa menjadi fitur terstruktur. Fitur ini penting agar recommender v2 tidak hanya bergantung pada kemiripan kata, tetapi juga pada profil minat, kecerdasan, gaya belajar, ketahanan belajar, dan orientasi karier.

In [9]:
# 9. Taxonomy RIASEC

riasec_taxonomy = pd.DataFrame([
    ['R', 'Realistic', 'Suka aktivitas praktis, alat, mesin, lapangan, teknis, dan problem fisik.', 'praktik|alat|mesin|lapangan|teknik|jaringan|bangunan|pertanian|robotik', 'Teknik Mesin; Teknik Sipil; Teknologi Informasi; Agroteknologi'],
    ['I', 'Investigative', 'Suka analisis, riset, eksperimen, sains, data, dan pemecahan masalah konseptual.', 'analisis|riset|eksperimen|data|statistik|sains|laboratorium|matematika', 'Sains Data; Statistika; Kedokteran; Farmasi; Kimia'],
    ['A', 'Artistic', 'Suka ekspresi kreatif, desain, visual, musik, menulis, storytelling, dan karya bebas.', 'desain|gambar|musik|menulis|konten|kreatif|film|visual|seni', 'DKV; Desain Produk; Film dan Televisi; Ilmu Komunikasi'],
    ['S', 'Social', 'Suka membantu, mengajar, konseling, pelayanan, kolaborasi, dan interaksi sosial.', 'membantu|mengajar|konseling|komunitas|pasien|anak|pelayanan|edukasi', 'Keperawatan; Psikologi; PGSD; Kesehatan Masyarakat'],
    ['E', 'Enterprising', 'Suka memimpin, bisnis, negosiasi, persuasi, organisasi, dan mengambil peluang.', 'bisnis|jualan|memimpin|presentasi|negosiasi|startup|marketing|event', 'Manajemen; Bisnis Digital; Ilmu Komunikasi; Pariwisata'],
    ['C', 'Conventional', 'Suka struktur, administrasi, angka teratur, laporan, dokumentasi, dan prosedur.', 'rapi|administrasi|akuntansi|dokumen|laporan|pajak|arsip|prosedur', 'Akuntansi; Sistem Informasi; Administrasi Publik; Ilmu Hukum'],
], columns=['riasec_code', 'riasec_name', 'description', 'keyword_signals', 'example_programs'])

riasec_output = TAXONOMY_DIR / 'riasec_taxonomy_v2.csv'
riasec_taxonomy.to_csv(riasec_output, index=False)

display(riasec_taxonomy)
print('Saved:', riasec_output)

,riasec_code,riasec_name,description,keyword_signals,example_programs
0,R,Realistic,"Suka aktivitas praktis, alat, mesin, lapangan, teknis, dan problem fisik.",praktik|alat|mesin|lapangan|teknik|jaringan|bangunan|pertanian|robotik,Teknik Mesin; Teknik Sipil; Teknologi Informasi; Agroteknologi
1,I,Investigative,"Suka analisis, riset, eksperimen, sains, data, dan pemecahan masalah konseptual.",analisis|riset|eksperimen|data|statistik|sains|laboratorium|matematika,Sains Data; Statistika; Kedokteran; Farmasi; Kimia
2,A,Artistic,"Suka ekspresi kreatif, desain, visual, musik, menulis, storytelling, dan karya bebas.",desain|gambar|musik|menulis|konten|kreatif|film|visual|seni,DKV; Desain Produk; Film dan Televisi; Ilmu Komunikasi
3,S,Social,"Suka membantu, mengajar, konseling, pelayanan, kolaborasi, dan interaksi sosial.",membantu|mengajar|konseling|komunitas|pasien|anak|pelayanan|edukasi,Keperawatan; Psikologi; PGSD; Kesehatan Masyarakat
4,E,Enterprising,"Suka memimpin, bisnis, negosiasi, persuasi, organisasi, dan mengambil peluang.",bisnis|jualan|memimpin|presentasi|negosiasi|startup|marketing|event,Manajemen; Bisnis Digital; Ilmu Komunikasi; Pariwisata
5,C,Conventional,"Suka struktur, administrasi, angka teratur, laporan, dokumentasi, dan prosedur.",rapi|administrasi|akuntansi|dokumen|laporan|pajak|arsip|prosedur,Akuntansi; Sistem Informasi; Administrasi Publik; Ilmu Hukum


Saved: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\data\taxonomy\riasec_taxonomy_v2.csv


In [10]:
# Code Cell — 10. Taxonomy Multiple Intelligences

mi_taxonomy = pd.DataFrame([
    ['MI-001', 'Linguistik', 'Kuat dalam bahasa, membaca, menulis, berbicara, argumentasi.', 'menulis|membaca|bahasa|pidato|debat|cerita|komunikasi', 'Hukum; Ilmu Komunikasi; Pendidikan Bahasa Inggris'],
    ['MI-002', 'Visual-Spasial', 'Kuat dalam visual, ruang, gambar, desain, peta, bentuk.', 'desain|gambar|visual|ruang|sketsa|layout|arsitektur', 'Arsitektur; DKV; Desain Produk'],
    ['MI-003', 'Musik', 'Kuat dalam ritme, nada, komposisi, audio, musik.', 'musik|lagu|audio|ritme|aransemen|sound', 'Seni Musik; Film dan Televisi; Ilmu Komunikasi'],
    ['MI-004', 'Interpersonal', 'Kuat dalam memahami orang lain, kolaborasi, komunikasi, pelayanan.', 'teman|kelompok|membantu|konseling|komunikasi|mengajar', 'Psikologi; Keperawatan; Manajemen; PGSD'],
    ['MI-005', 'Logis-Matematis', 'Kuat dalam angka, logika, pola, algoritma, analisis, pemodelan.', 'matematika|logika|angka|algoritma|statistik|data|rumus', 'Sains Data; Statistika; Teknik Informatika; Akuntansi'],
    ['MI-006', 'Kinestetik', 'Kuat dalam aktivitas fisik, praktik langsung, eksperimen, gerak, simulasi.', 'praktik|gerak|membuat|merakit|lapangan|eksperimen|olahraga', 'Teknik Mesin; Keperawatan; Agroteknologi; Teknik Sipil'],
    ['MI-007', 'Intrapersonal', 'Kuat dalam refleksi diri, tujuan pribadi, disiplin, pemahaman emosi diri.', 'refleksi|mandiri|target|introspeksi|motivasi|fokus', 'Psikologi; Filsafat; Manajemen; Pendidikan'],
    ['MI-008', 'Naturalis', 'Kuat dalam alam, hewan, tumbuhan, lingkungan, pangan, kesehatan ekologis.', 'alam|hewan|tumbuhan|lingkungan|pangan|pertanian|biologi', 'Biologi; Agroteknologi; Agribisnis; Teknologi Pangan'],
], columns=['mi_id', 'mi_name', 'description', 'keyword_signals', 'example_programs'])

mi_output = TAXONOMY_DIR / 'multiple_intelligences_taxonomy_v2.csv'
mi_taxonomy.to_csv(mi_output, index=False)

display(mi_taxonomy)
print('Saved:', mi_output)

,mi_id,mi_name,description,keyword_signals,example_programs
0,MI-001,Linguistik,"Kuat dalam bahasa, membaca, menulis, berbicara, argumentasi.",menulis|membaca|bahasa|pidato|debat|cerita|komunikasi,Hukum; Ilmu Komunikasi; Pendidikan Bahasa Inggris
1,MI-002,Visual-Spasial,"Kuat dalam visual, ruang, gambar, desain, peta, bentuk.",desain|gambar|visual|ruang|sketsa|layout|arsitektur,Arsitektur; DKV; Desain Produk
2,MI-003,Musik,"Kuat dalam ritme, nada, komposisi, audio, musik.",musik|lagu|audio|ritme|aransemen|sound,Seni Musik; Film dan Televisi; Ilmu Komunikasi
3,MI-004,Interpersonal,"Kuat dalam memahami orang lain, kolaborasi, komunikasi, pelayanan.",teman|kelompok|membantu|konseling|komunikasi|mengajar,Psikologi; Keperawatan; Manajemen; PGSD
4,MI-005,Logis-Matematis,"Kuat dalam angka, logika, pola, algoritma, analisis, pemodelan.",matematika|logika|angka|algoritma|statistik|data|rumus,Sains Data; Statistika; Teknik Informatika; Akuntansi
5,MI-006,Kinestetik,"Kuat dalam aktivitas fisik, praktik langsung, eksperimen, gerak, simulasi.",praktik|gerak|membuat|merakit|lapangan|eksperimen|olahraga,Teknik Mesin; Keperawatan; Agroteknologi; Teknik Sipil
6,MI-007,Intrapersonal,"Kuat dalam refleksi diri, tujuan pribadi, disiplin, pemahaman emosi diri.",refleksi|mandiri|target|introspeksi|motivasi|fokus,Psikologi; Filsafat; Manajemen; Pendidikan
7,MI-008,Naturalis,"Kuat dalam alam, hewan, tumbuhan, lingkungan, pangan, kesehatan ekologis.",alam|hewan|tumbuhan|lingkungan|pangan|pertanian|biologi,Biologi; Agroteknologi; Agribisnis; Teknologi Pangan


Saved: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\data\taxonomy\multiple_intelligences_taxonomy_v2.csv


In [11]:
# 11. Taxonomy VARK

vark_taxonomy = pd.DataFrame([
    ['VARK-V', 'Visual', 'Lebih mudah memahami melalui diagram, grafik, peta konsep, visualisasi, dan contoh visual.', 'grafik|diagram|visual|gambar|peta konsep|warna|dashboard', 'Gunakan infografik, flowchart, mindmap, dashboard, dan video visual.'],
    ['VARK-R', 'Reading/Writing', 'Lebih mudah memahami melalui membaca, membuat catatan, rangkuman, dan tulisan terstruktur.', 'membaca|menulis|catatan|rangkuman|artikel|buku|teks', 'Gunakan modul, rangkuman, checklist, dokumentasi, dan latihan tertulis.'],
    ['VARK-A', 'Auditory', 'Lebih mudah memahami melalui diskusi, mendengarkan, penjelasan verbal, dan presentasi.', 'diskusi|dengar|podcast|presentasi|cerita|menjelaskan', 'Gunakan diskusi, rekaman audio, presentasi, dan peer teaching.'],
    ['VARK-K', 'Kinesthetic', 'Lebih mudah memahami melalui praktik langsung, simulasi, eksperimen, proyek, dan pengalaman nyata.', 'praktik|simulasi|proyek|eksperimen|langsung|mencoba|lapangan', 'Gunakan project-based learning, studi kasus, lab, praktik, dan portofolio.'],
], columns=['vark_id', 'vark_name', 'description', 'keyword_signals', 'learning_strategy'])

vark_output = TAXONOMY_DIR / 'vark_taxonomy_v2.csv'
vark_taxonomy.to_csv(vark_output, index=False)

display(vark_taxonomy)
print('Saved:', vark_output)

,vark_id,vark_name,description,keyword_signals,learning_strategy
0,VARK-V,Visual,"Lebih mudah memahami melalui diagram, grafik, peta konsep, visualisasi, dan contoh visual.",grafik|diagram|visual|gambar|peta konsep|warna|dashboard,"Gunakan infografik, flowchart, mindmap, dashboard, dan video visual."
1,VARK-R,Reading/Writing,"Lebih mudah memahami melalui membaca, membuat catatan, rangkuman, dan tulisan terstruktur.",membaca|menulis|catatan|rangkuman|artikel|buku|teks,"Gunakan modul, rangkuman, checklist, dokumentasi, dan latihan tertulis."
2,VARK-A,Auditory,"Lebih mudah memahami melalui diskusi, mendengarkan, penjelasan verbal, dan presentasi.",diskusi|dengar|podcast|presentasi|cerita|menjelaskan,"Gunakan diskusi, rekaman audio, presentasi, dan peer teaching."
3,VARK-K,Kinesthetic,"Lebih mudah memahami melalui praktik langsung, simulasi, eksperimen, proyek, dan pengalaman nyata.",praktik|simulasi|proyek|eksperimen|langsung|mencoba|lapangan,"Gunakan project-based learning, studi kasus, lab, praktik, dan portofolio."


Saved: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\data\taxonomy\vark_taxonomy_v2.csv


In [12]:
# 12. Taxonomy Grit dan Mindset

grit_mindset_taxonomy = pd.DataFrame([
    ['GRIT-LOW', 'Low Grit', 'Mudah menyerah, sering kehilangan fokus, membutuhkan dukungan struktur belajar tinggi.', 'mudah menyerah|cepat bosan|tidak konsisten|bingung mulai|takut gagal', 'Berikan roadmap kecil, target mingguan, mentor/guru BK, dan milestone rendah risiko.'],
    ['GRIT-MED', 'Medium Grit', 'Cukup mampu bertahan, tetapi perlu penguatan konsistensi dan feedback rutin.', 'kadang konsisten|butuh arahan|ingin belajar|perlu jadwal', 'Berikan target 3 bulan, latihan bertahap, evaluasi mingguan, dan project sederhana.'],
    ['GRIT-HIGH', 'High Grit', 'Mampu bertahan, disiplin, dan siap menjalani roadmap yang lebih menantang.', 'konsisten|pantang menyerah|target jelas|rajin latihan|disiplin', 'Berikan roadmap intensif, project portofolio, kompetisi, dan eksplorasi lanjut.'],
    ['MIND-GROWTH', 'Growth Mindset', 'Percaya kemampuan dapat berkembang melalui latihan, strategi, dan feedback.', 'bisa belajar|ingin berkembang|mau mencoba|belajar dari salah|latihan', 'Cocok dengan rekomendasi prodi menantang selama roadmap jelas.'],
    ['MIND-FIXED', 'Fixed Mindset', 'Cenderung merasa kemampuan tetap dan takut gagal pada bidang sulit.', 'tidak bakat|takut gagal|saya tidak bisa|bukan anak pintar|sulit berubah', 'Gunakan respons suportif, validasi emosi, dan berikan langkah awal ringan.'],
    ['MIND-MIXED', 'Mixed Mindset', 'Memiliki kombinasi keyakinan berkembang dan kekhawatiran tertentu.', 'mau mencoba tapi takut|ingin belajar tapi ragu|kadang percaya diri', 'Berikan rekomendasi alternatif dan bridge plan.'],
], columns=['mindset_id', 'mindset_name', 'description', 'keyword_signals', 'recommended_response_strategy'])

grit_output = TAXONOMY_DIR / 'grit_mindset_taxonomy_v2.csv'
grit_mindset_taxonomy.to_csv(grit_output, index=False)

display(grit_mindset_taxonomy)
print('Saved:', grit_output)

,mindset_id,mindset_name,description,keyword_signals,recommended_response_strategy
0,GRIT-LOW,Low Grit,"Mudah menyerah, sering kehilangan fokus, membutuhkan dukungan struktur belajar tinggi.",mudah menyerah|cepat bosan|tidak konsisten|bingung mulai|takut gagal,"Berikan roadmap kecil, target mingguan, mentor/guru BK, dan milestone rendah risiko."
1,GRIT-MED,Medium Grit,"Cukup mampu bertahan, tetapi perlu penguatan konsistensi dan feedback rutin.",kadang konsisten|butuh arahan|ingin belajar|perlu jadwal,"Berikan target 3 bulan, latihan bertahap, evaluasi mingguan, dan project sederhana."
2,GRIT-HIGH,High Grit,"Mampu bertahan, disiplin, dan siap menjalani roadmap yang lebih menantang.",konsisten|pantang menyerah|target jelas|rajin latihan|disiplin,"Berikan roadmap intensif, project portofolio, kompetisi, dan eksplorasi lanjut."
3,MIND-GROWTH,Growth Mindset,"Percaya kemampuan dapat berkembang melalui latihan, strategi, dan feedback.",bisa belajar|ingin berkembang|mau mencoba|belajar dari salah|latihan,Cocok dengan rekomendasi prodi menantang selama roadmap jelas.
4,MIND-FIXED,Fixed Mindset,Cenderung merasa kemampuan tetap dan takut gagal pada bidang sulit.,tidak bakat|takut gagal|saya tidak bisa|bukan anak pintar|sulit berubah,"Gunakan respons suportif, validasi emosi, dan berikan langkah awal ringan."
5,MIND-MIXED,Mixed Mindset,Memiliki kombinasi keyakinan berkembang dan kekhawatiran tertentu.,mau mencoba tapi takut|ingin belajar tapi ragu|kadang percaya diri,Berikan rekomendasi alternatif dan bridge plan.


Saved: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\data\taxonomy\grit_mindset_taxonomy_v2.csv


In [13]:
# 13. Taxonomy Career Alignment

career_alignment_taxonomy = pd.DataFrame([
    ['CAR-ACA', 'Akademik', 'Berorientasi riset, teori, analisis mendalam, pendidikan lanjut, dan pengembangan ilmu.', 'riset|penelitian|akademik|teori|data|laboratorium|kuliah lanjut', 'Sains Data; Statistika; Kedokteran; Biologi; Kimia'],
    ['CAR-SOC', 'Sosial', 'Berorientasi membantu orang, layanan, edukasi, kesehatan, komunitas, dan dampak sosial.', 'membantu|masyarakat|mengajar|pasien|konseling|komunitas|pelayanan', 'Keperawatan; Psikologi; PGSD; Kesehatan Masyarakat'],
    ['CAR-PRA', 'Praktikal', 'Berorientasi praktik, teknik, troubleshooting, produksi, lapangan, dan implementasi langsung.', 'praktik|teknis|lapangan|mesin|jaringan|membangun|produksi', 'Teknik Mesin; Teknik Sipil; Teknologi Informasi; Agroteknologi'],
    ['CAR-KRE', 'Kreatif', 'Berorientasi ide, desain, visual, konten, media, seni, dan produk kreatif.', 'kreatif|desain|konten|video|musik|brand|visual|storytelling', 'DKV; Desain Produk; Film dan Televisi; Ilmu Komunikasi'],
    ['CAR-EKS', 'Eksekutif', 'Berorientasi bisnis, kepemimpinan, manajemen, organisasi, strategi, dan pengambilan keputusan.', 'bisnis|memimpin|manajemen|strategi|jualan|organisasi|negosiasi', 'Manajemen; Bisnis Digital; Akuntansi; Administrasi Publik'],
], columns=['career_alignment_id', 'career_alignment_name', 'description', 'keyword_signals', 'example_programs'])

career_output = TAXONOMY_DIR / 'career_alignment_taxonomy_v2.csv'
career_alignment_taxonomy.to_csv(career_output, index=False)

display(career_alignment_taxonomy)
print('Saved:', career_output)

,career_alignment_id,career_alignment_name,description,keyword_signals,example_programs
0,CAR-ACA,Akademik,"Berorientasi riset, teori, analisis mendalam, pendidikan lanjut, dan pengembangan ilmu.",riset|penelitian|akademik|teori|data|laboratorium|kuliah lanjut,Sains Data; Statistika; Kedokteran; Biologi; Kimia
1,CAR-SOC,Sosial,"Berorientasi membantu orang, layanan, edukasi, kesehatan, komunitas, dan dampak sosial.",membantu|masyarakat|mengajar|pasien|konseling|komunitas|pelayanan,Keperawatan; Psikologi; PGSD; Kesehatan Masyarakat
2,CAR-PRA,Praktikal,"Berorientasi praktik, teknik, troubleshooting, produksi, lapangan, dan implementasi langsung.",praktik|teknis|lapangan|mesin|jaringan|membangun|produksi,Teknik Mesin; Teknik Sipil; Teknologi Informasi; Agroteknologi
3,CAR-KRE,Kreatif,"Berorientasi ide, desain, visual, konten, media, seni, dan produk kreatif.",kreatif|desain|konten|video|musik|brand|visual|storytelling,DKV; Desain Produk; Film dan Televisi; Ilmu Komunikasi
4,CAR-EKS,Eksekutif,"Berorientasi bisnis, kepemimpinan, manajemen, organisasi, strategi, dan pengambilan keputusan.",bisnis|memimpin|manajemen|strategi|jualan|organisasi|negosiasi,Manajemen; Bisnis Digital; Akuntansi; Administrasi Publik


Saved: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\data\taxonomy\career_alignment_taxonomy_v2.csv


## 11. Data Dictionary untuk Semua Taxonomy

Data dictionary berikut menjadi pedoman agar setiap CSV taxonomy dapat dibaca dan dikembangkan secara konsisten pada tahap berikutnya.

In [14]:
# 14. Data Dictionary Taxonomy v2

data_dictionary_rows = [
    ['school_taxonomy_v2', 'school_taxonomy_id', 'string', 'Primary key internal taxonomy sekolah.', 'SCH-SMK-001'],
    ['school_taxonomy_v2', 'school_type', 'category', 'Jenis satuan pendidikan menengah.', 'SMA/MA/SMK/MAK'],
    ['school_taxonomy_v2', 'school_group', 'category', 'Kelompok umum, madrasah umum, vokasi, atau madrasah vokasi.', 'Vokasi'],
    ['school_taxonomy_v2', 'school_major_detail', 'string', 'Jurusan, kelompok mapel, atau konsentrasi keahlian siswa.', 'RPL'],
    ['school_taxonomy_v2', 'subject_context', 'string', 'Konteks mata pelajaran atau kompetensi utama.', 'Pemrograman, database, web'],
    ['school_taxonomy_v2', 'recommended_higher_education_area', 'string', 'Area pendidikan tinggi yang umumnya relevan.', 'Teknik Informatika, Sistem Informasi'],
    ['program_studi_taxonomy_pddikti_style_v2', 'program_id', 'string', 'Primary key internal program studi.', 'PRG2-001'],
    ['program_studi_taxonomy_pddikti_style_v2', 'program_name', 'string', 'Nama program studi S1.', 'Teknik Informatika'],
    ['program_studi_taxonomy_pddikti_style_v2', 'jenjang', 'category', 'Jenjang pendidikan tinggi.', 'S1'],
    ['program_studi_taxonomy_pddikti_style_v2', 'rumpun_ilmu', 'string', 'Rumpun ilmu atau bidang besar.', 'Komputer/Informatika'],
    ['program_studi_taxonomy_pddikti_style_v2', 'riasec_primary', 'category', 'Kode/nama RIASEC utama untuk program.', 'Investigative'],
    ['program_studi_taxonomy_pddikti_style_v2', 'riasec_secondary', 'category', 'Kode/nama RIASEC sekunder.', 'Realistic'],
    ['program_studi_taxonomy_pddikti_style_v2', 'mi_primary', 'category', 'Kecerdasan dominan yang paling relevan.', 'Logis-Matematis'],
    ['program_studi_taxonomy_pddikti_style_v2', 'vark_preferred', 'category', 'Gaya belajar yang paling mendukung.', 'Kinesthetic'],
    ['program_studi_taxonomy_pddikti_style_v2', 'career_alignment', 'category', 'Orientasi karier utama.', 'Praktikal'],
    ['program_studi_taxonomy_pddikti_style_v2', 'school_fit', 'string', 'Latar sekolah/jurusan yang relatif relevan.', 'SMA MIPA; SMK RPL/TKJ'],
    ['program_studi_taxonomy_pddikti_style_v2', 'pddikti_reference_status', 'category', 'Status validasi terhadap referensi PDDikti.', 'prototype_pddikti_style'],
    ['riasec_taxonomy_v2', 'riasec_code', 'category', 'Kode Holland RIASEC.', 'R/I/A/S/E/C'],
    ['riasec_taxonomy_v2', 'keyword_signals', 'string', 'Keyword pemicu dari input siswa.', 'coding|data|praktik'],
    ['multiple_intelligences_taxonomy_v2', 'mi_id', 'string', 'Primary key Multiple Intelligences.', 'MI-005'],
    ['vark_taxonomy_v2', 'vark_id', 'string', 'Primary key VARK.', 'VARK-V'],
    ['grit_mindset_taxonomy_v2', 'mindset_id', 'string', 'Primary key grit/mindset.', 'MIND-GROWTH'],
    ['career_alignment_taxonomy_v2', 'career_alignment_id', 'string', 'Primary key career alignment.', 'CAR-PRA'],
]

data_dictionary = pd.DataFrame(data_dictionary_rows, columns=['taxonomy_file', 'field_name', 'data_type', 'description', 'example_value'])

dict_output = REPORT_STAGE09_DIR / 'taxonomy_data_dictionary_v2.csv'
data_dictionary.to_csv(dict_output, index=False)

display(data_dictionary)
print('Saved:', dict_output)

,taxonomy_file,field_name,data_type,description,example_value
0,school_taxonomy_v2,school_taxonomy_id,string,Primary key internal taxonomy sekolah.,SCH-SMK-001
1,school_taxonomy_v2,school_type,category,Jenis satuan pendidikan menengah.,SMA/MA/SMK/MAK
2,school_taxonomy_v2,school_group,category,"Kelompok umum, madrasah umum, vokasi, atau madrasah vokasi.",Vokasi
3,school_taxonomy_v2,school_major_detail,string,"Jurusan, kelompok mapel, atau konsentrasi keahlian siswa.",RPL
4,school_taxonomy_v2,subject_context,string,Konteks mata pelajaran atau kompetensi utama.,"Pemrograman, database, web"
5,school_taxonomy_v2,recommended_higher_education_area,string,Area pendidikan tinggi yang umumnya relevan.,"Teknik Informatika, Sistem Informasi"
6,program_studi_taxonomy_pddikti_style_v2,program_id,string,Primary key internal program studi.,PRG2-001
7,program_studi_taxonomy_pddikti_style_v2,program_name,string,Nama program studi S1.,Teknik Informatika
8,program_studi_taxonomy_pddikti_style_v2,jenjang,category,Jenjang pendidikan tinggi.,S1
9,program_studi_taxonomy_pddikti_style_v2,rumpun_ilmu,string,Rumpun ilmu atau bidang besar.,Komputer/Informatika


Saved: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\reports\stage09\taxonomy_data_dictionary_v2.csv


## 12. Rancangan Struktur Dataset v2

Dataset v2 sebaiknya dipisahkan menjadi beberapa tabel agar lebih maintainable:

1. `student_profile_synthetic_v2.csv` — profil sintetis siswa.
2. `chat_history_synthetic_v2.csv` — variasi percakapan siswa.
3. `intent_dataset_synthetic_v2.csv` — dataset intent classification.
4. `program_studi_taxonomy_pddikti_style_v2.csv` — katalog prodi S1.
5. `recommendation_training_pairs_v2.csv` — pasangan profil siswa dan label rekomendasi expected.
6. `roadmap_taxonomy_v2.csv` — roadmap belajar per program/rumpun.
7. `evaluation_scenarios_v2.csv` — scenario testing chatbot.

Cell berikut membuat skema kolom sebagai acuan tahap berikutnya.

In [15]:
# 15. Rancangan Struktur Dataset v2

dataset_schema = pd.DataFrame([
    ['student_profile_synthetic_v2.csv', 'student_id', 'string', 'Primary key siswa sintetis.', 'STU-000001'],
    ['student_profile_synthetic_v2.csv', 'school_type', 'category', 'SMA/MA/SMK/MAK.', 'SMK'],
    ['student_profile_synthetic_v2.csv', 'school_major_detail', 'string', 'Jurusan/konsentrasi/kelompok mapel.', 'RPL'],
    ['student_profile_synthetic_v2.csv', 'subject_strengths', 'string', 'Mapel/kompetensi dominan dipisahkan pipe.', 'Matematika|Informatika|Basis Data'],
    ['student_profile_synthetic_v2.csv', 'hobby_signals', 'string', 'Hobi/minat aktivitas.', 'ngoding|membuat aplikasi'],
    ['student_profile_synthetic_v2.csv', 'riasec_primary', 'category', 'RIASEC utama hasil synthetic rule.', 'Investigative'],
    ['student_profile_synthetic_v2.csv', 'riasec_secondary', 'category', 'RIASEC sekunder.', 'Realistic'],
    ['student_profile_synthetic_v2.csv', 'mi_primary', 'category', 'Multiple Intelligence dominan.', 'Logis-Matematis'],
    ['student_profile_synthetic_v2.csv', 'vark_primary', 'category', 'Gaya belajar dominan.', 'Kinesthetic'],
    ['student_profile_synthetic_v2.csv', 'grit_level', 'category', 'Low/Medium/High.', 'Medium'],
    ['student_profile_synthetic_v2.csv', 'mindset_type', 'category', 'Growth/Fixed/Mixed.', 'Growth'],
    ['student_profile_synthetic_v2.csv', 'career_alignment', 'category', 'Akademik/Sosial/Praktikal/Kreatif/Eksekutif.', 'Praktikal'],
    ['student_profile_synthetic_v2.csv', 'initial_choice_program', 'string', 'Jurusan awal pilihan siswa.', 'Pendidikan Dokter'],
    ['student_profile_synthetic_v2.csv', 'expected_top_programs', 'string', 'Label expected Top-N dipisahkan pipe.', 'Teknik Informatika|Sains Data|Sistem Informasi'],
    ['chat_history_synthetic_v2.csv', 'conversation_id', 'string', 'Primary key percakapan.', 'CONV-000001'],
    ['chat_history_synthetic_v2.csv', 'student_id', 'string', 'Foreign key ke student_profile.', 'STU-000001'],
    ['chat_history_synthetic_v2.csv', 'turn_id', 'integer', 'Urutan percakapan.', '1'],
    ['chat_history_synthetic_v2.csv', 'speaker', 'category', 'user/bot.', 'user'],
    ['chat_history_synthetic_v2.csv', 'message_text', 'string', 'Isi pesan.', 'Aku suka coding dan bingung pilih TI atau Sains Data'],
    ['chat_history_synthetic_v2.csv', 'language_variant', 'category', 'id_formal/id_nonformal/jawa_umum/sunda_umum/mixed.', 'id_nonformal'],
    ['intent_dataset_synthetic_v2.csv', 'utterance_id', 'string', 'Primary key utterance.', 'UTT2-000001'],
    ['intent_dataset_synthetic_v2.csv', 'utterance', 'string', 'Teks input user.', 'Saya mau bandingkan Kedokteran dengan profil saya'],
    ['intent_dataset_synthetic_v2.csv', 'intent_label', 'category', 'Label intent.', 'bandingkan_pilihan_awal'],
    ['intent_dataset_synthetic_v2.csv', 'source_type', 'category', 'synthetic_template/synthetic_paraphrase/manual_seed/off_domain.', 'synthetic_template'],
    ['intent_dataset_synthetic_v2.csv', 'expected_entities', 'json/string', 'Entity penting hasil ekstraksi.', '{"initial_choice":"Pendidikan Dokter"}'],
    ['recommendation_training_pairs_v2.csv', 'pair_id', 'string', 'Primary key pair.', 'PAIR-000001'],
    ['recommendation_training_pairs_v2.csv', 'student_id', 'string', 'Foreign key siswa.', 'STU-000001'],
    ['recommendation_training_pairs_v2.csv', 'program_id', 'string', 'Foreign key program studi.', 'PRG2-001'],
    ['recommendation_training_pairs_v2.csv', 'label_relevance', 'integer', '0 tidak relevan, 1 cukup, 2 relevan, 3 sangat relevan.', '3'],
    ['evaluation_scenarios_v2.csv', 'scenario_id', 'string', 'Primary key skenario.', 'SCN2-001'],
    ['evaluation_scenarios_v2.csv', 'scenario_group', 'category', 'rekomendasi/prospek/roadmap/fallback/initial_choice.', 'initial_choice'],
    ['evaluation_scenarios_v2.csv', 'expected_behavior', 'string', 'Kriteria respons yang diharapkan.', 'Membandingkan pilihan awal dengan Top-N hasil mapping'],
], columns=['dataset_name', 'field_name', 'data_type', 'description', 'example_value'])

schema_output = REPORT_STAGE09_DIR / 'dataset_schema_v2.csv'
dataset_schema.to_csv(schema_output, index=False)

display(dataset_schema)
print('Saved:', schema_output)

,dataset_name,field_name,data_type,description,example_value
0,student_profile_synthetic_v2.csv,student_id,string,Primary key siswa sintetis.,STU-000001
1,student_profile_synthetic_v2.csv,school_type,category,SMA/MA/SMK/MAK.,SMK
2,student_profile_synthetic_v2.csv,school_major_detail,string,Jurusan/konsentrasi/kelompok mapel.,RPL
3,student_profile_synthetic_v2.csv,subject_strengths,string,Mapel/kompetensi dominan dipisahkan pipe.,Matematika|Informatika|Basis Data
4,student_profile_synthetic_v2.csv,hobby_signals,string,Hobi/minat aktivitas.,ngoding|membuat aplikasi
5,student_profile_synthetic_v2.csv,riasec_primary,category,RIASEC utama hasil synthetic rule.,Investigative
6,student_profile_synthetic_v2.csv,riasec_secondary,category,RIASEC sekunder.,Realistic
7,student_profile_synthetic_v2.csv,mi_primary,category,Multiple Intelligence dominan.,Logis-Matematis
8,student_profile_synthetic_v2.csv,vark_primary,category,Gaya belajar dominan.,Kinesthetic
9,student_profile_synthetic_v2.csv,grit_level,category,Low/Medium/High.,Medium


Saved: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\reports\stage09\dataset_schema_v2.csv


## 13. Rancangan Scoring Engine dan Recommender Logic Awal

Recommender v2 menggunakan pendekatan hybrid agar tidak hanya bergantung pada TF-IDF. Komponen skor disusun sebagai berikut:

\[
FinalScore = 0.25(TextSimilarity) + 0.20(RIASECFit) + 0.15(MIFit) + 0.10(SchoolFit) + 0.10(CareerFit) + 0.10(AcademicReadiness) + 0.05(VARKFit) + 0.05(InitialChoiceAlignment)
\]

Interpretasi awal:

- **TextSimilarity:** kecocokan teks profil siswa dengan profil prodi.
- **RIASECFit:** kecocokan minat/passion siswa dengan RIASEC prodi.
- **MIFit:** kecocokan kecerdasan dominan dengan tuntutan prodi.
- **SchoolFit:** kecocokan latar sekolah/jurusan/konsentrasi.
- **CareerFit:** keselarasan orientasi karier.
- **AcademicReadiness:** readiness akademik berdasarkan subject_strengths.
- **VARKFit:** kecocokan gaya belajar dengan karakter pembelajaran prodi.
- **InitialChoiceAlignment:** apakah jurusan awal siswa selaras dengan hasil scoring.

Skor ini masih rancangan awal dan perlu dikalibrasi dengan evaluasi domain expert.

In [16]:
# 16. Konfigurasi Scoring Engine v2

scoring_config = {
    'scoring_engine_version': 'v2.0-stage09-design',
    'score_components': {
        'text_similarity': 0.25,
        'riasec_fit': 0.20,
        'multiple_intelligence_fit': 0.15,
        'school_background_fit': 0.10,
        'career_alignment_fit': 0.10,
        'academic_readiness': 0.10,
        'vark_fit': 0.05,
        'initial_choice_alignment': 0.05
    },
    'score_interpretation': {
        '0.80-1.00': 'Sangat sesuai sebagai prioritas utama eksplorasi.',
        '0.65-0.79': 'Cukup sesuai, tetapi perlu penguatan kompetensi tertentu.',
        '0.50-0.64': 'Masih mungkin, namun perlu bridge plan dan validasi tambahan.',
        '<0.50': 'Tidak menjadi prioritas awal; dapat dipertimbangkan sebagai opsi alternatif setelah eksplorasi.'
    },
    'guardrails': [
        'Rekomendasi bukan keputusan final.',
        'Tidak memberikan diagnosis psikologis.',
        'Tidak menolak mimpi siswa secara mutlak; gunakan bahasa suportif dan bridge plan.',
        'Sarankan validasi dengan guru BK/orang tua/konselor akademik.'
    ]
}

scoring_config_output = MODEL_DIR / 'recommender_scoring_config_v2.json'
with open(scoring_config_output, 'w', encoding='utf-8') as f:
    json.dump(scoring_config, f, ensure_ascii=False, indent=2)

print(json.dumps(scoring_config, ensure_ascii=False, indent=2))
print('Saved:', scoring_config_output)

{
  "scoring_engine_version": "v2.0-stage09-design",
  "score_components": {
    "text_similarity": 0.25,
    "riasec_fit": 0.2,
    "multiple_intelligence_fit": 0.15,
    "school_background_fit": 0.1,
    "career_alignment_fit": 0.1,
    "academic_readiness": 0.1,
    "vark_fit": 0.05,
    "initial_choice_alignment": 0.05
  },
  "score_interpretation": {
    "0.80-1.00": "Sangat sesuai sebagai prioritas utama eksplorasi.",
    "0.65-0.79": "Cukup sesuai, tetapi perlu penguatan kompetensi tertentu.",
    "0.50-0.64": "Masih mungkin, namun perlu bridge plan dan validasi tambahan.",
    "<0.50": "Tidak menjadi prioritas awal; dapat dipertimbangkan sebagai opsi alternatif setelah eksplorasi."
  },
  "guardrails": [
    "Rekomendasi bukan keputusan final.",
    "Tidak memberikan diagnosis psikologis.",
    "Tidak menolak mimpi siswa secara mutlak; gunakan bahasa suportif dan bridge plan.",
    "Sarankan validasi dengan guru BK/orang tua/konselor akademik."
  ]
}
Saved: D:\DATA-KAMIL\MATKUL

In [17]:
# 17. Simulasi Sederhana Scoring Engine v2

# Contoh profil siswa: siswa ingin Kedokteran, tetapi sinyal kuatnya coding/data.
sample_student_profile = {
    'student_id': 'STU-SAMPLE-001',
    'school_type': 'SMK',
    'school_major_detail': 'RPL',
    'subject_strengths': ['Matematika', 'Informatika', 'Basis Data'],
    'hobby_signals': ['ngoding', 'membuat dashboard', 'analisis data'],
    'riasec_primary': 'Investigative',
    'riasec_secondary': 'Realistic',
    'mi_primary': 'Logis-Matematis',
    'vark_primary': 'Kinesthetic',
    'grit_level': 'Medium',
    'mindset_type': 'Growth',
    'career_alignment': 'Praktikal',
    'initial_choice_program': 'Pendidikan Dokter',
    'free_text': 'Saya anak SMK RPL, suka coding, matematika, data, tapi awalnya ingin masuk Kedokteran karena keluarga menyarankan.'
}

weights = scoring_config['score_components']

# Fungsi utilitas sederhana untuk desain awal.
def contains_any(text, keyword_blob):
    text = str(text).lower()
    keywords = [k.strip().lower() for k in str(keyword_blob).replace(';', '|').split('|') if k.strip()]
    return any(k in text for k in keywords)

def simple_component_scores(student, program):
    profile_text = ' '.join([
        student.get('school_type', ''),
        student.get('school_major_detail', ''),
        ' '.join(student.get('subject_strengths', [])),
        ' '.join(student.get('hobby_signals', [])),
        student.get('free_text', '')
    ]).lower()
    
    text_similarity = 1.0 if contains_any(profile_text, program['keywords']) else 0.3
    riasec_fit = 1.0 if student['riasec_primary'] in [program['riasec_primary'], program['riasec_secondary']] else 0.5 if student['riasec_secondary'] in [program['riasec_primary'], program['riasec_secondary']] else 0.2
    mi_fit = 1.0 if student['mi_primary'] == program['mi_primary'] else 0.4
    school_fit = 1.0 if contains_any(student['school_major_detail'], program['school_fit']) or contains_any(program['school_fit'], student['school_major_detail']) else 0.5
    career_fit = 1.0 if student['career_alignment'] == program['career_alignment'] else 0.5
    academic_readiness = 0.8 if program['math_level'] in ['tinggi', 'sangat tinggi'] and 'Matematika' in student['subject_strengths'] else 0.6
    vark_fit = 1.0 if student['vark_primary'] == program['vark_preferred'] else 0.5
    initial_choice_alignment = 1.0 if student['initial_choice_program'].lower() == program['program_name'].lower() else 0.3
    
    final_score = (
        weights['text_similarity'] * text_similarity +
        weights['riasec_fit'] * riasec_fit +
        weights['multiple_intelligence_fit'] * mi_fit +
        weights['school_background_fit'] * school_fit +
        weights['career_alignment_fit'] * career_fit +
        weights['academic_readiness'] * academic_readiness +
        weights['vark_fit'] * vark_fit +
        weights['initial_choice_alignment'] * initial_choice_alignment
    )
    return {
        'text_similarity': round(text_similarity, 3),
        'riasec_fit': round(riasec_fit, 3),
        'multiple_intelligence_fit': round(mi_fit, 3),
        'school_background_fit': round(school_fit, 3),
        'career_alignment_fit': round(career_fit, 3),
        'academic_readiness': round(academic_readiness, 3),
        'vark_fit': round(vark_fit, 3),
        'initial_choice_alignment': round(initial_choice_alignment, 3),
        'final_score': round(final_score, 4)
    }

simulation_rows = []
for _, program in program_taxonomy.iterrows():
    scores = simple_component_scores(sample_student_profile, program)
    simulation_rows.append({
        'student_id': sample_student_profile['student_id'],
        'initial_choice_program': sample_student_profile['initial_choice_program'],
        'program_id': program['program_id'],
        'program_name': program['program_name'],
        **scores,
        'career_examples': program['career_examples']
    })

simulation_df = pd.DataFrame(simulation_rows).sort_values('final_score', ascending=False).reset_index(drop=True)
simulation_output = REPORT_STAGE09_DIR / 'sample_scoring_simulation_v2.csv'
simulation_df.to_csv(simulation_output, index=False)

display(simulation_df.head(10))
print('Saved:', simulation_output)

,student_id,initial_choice_program,program_id,program_name,text_similarity,riasec_fit,multiple_intelligence_fit,school_background_fit,career_alignment_fit,academic_readiness,vark_fit,initial_choice_alignment,final_score,career_examples
0,STU-SAMPLE-001,Pendidikan Dokter,PRG2-001,Teknik Informatika,1.0,1.0,1.0,1.0,1.0,0.8,1.0,0.3,0.945,Software Engineer; Backend Developer; AI Engineer
1,STU-SAMPLE-001,Pendidikan Dokter,PRG2-012,Statistika,1.0,1.0,1.0,1.0,0.5,0.8,0.5,0.3,0.870,Statistician; Data Analyst; Survey Researcher
2,STU-SAMPLE-001,Pendidikan Dokter,PRG2-003,Sains Data,1.0,1.0,1.0,1.0,0.5,0.8,0.5,0.3,0.870,Data Analyst; Data Scientist; ML Engineer
3,STU-SAMPLE-001,Pendidikan Dokter,PRG2-015,Pendidikan Dokter,1.0,1.0,1.0,0.5,0.5,0.8,0.5,1.0,0.855,Dokter; Clinical Researcher; Medical Educator
4,STU-SAMPLE-001,Pendidikan Dokter,PRG2-014,Kimia,1.0,1.0,1.0,0.5,0.5,0.8,1.0,0.3,0.845,Chemist; Lab Analyst; Quality Control Analyst
5,STU-SAMPLE-001,Pendidikan Dokter,PRG2-011,Matematika,1.0,1.0,1.0,0.5,0.5,0.8,0.5,0.3,0.820,Mathematician; Quant Analyst; Research Assistant
6,STU-SAMPLE-001,Pendidikan Dokter,PRG2-004,Teknologi Informasi,0.3,1.0,1.0,0.5,1.0,0.8,1.0,0.3,0.720,IT Infrastructure Engineer; Cloud Engineer; Network Engineer
7,STU-SAMPLE-001,Pendidikan Dokter,PRG2-006,Teknik Elektro,0.3,1.0,1.0,0.5,1.0,0.8,1.0,0.3,0.720,Electrical Engineer; IoT Engineer; Automation Engineer
8,STU-SAMPLE-001,Pendidikan Dokter,PRG2-008,Teknik Mesin,0.3,1.0,1.0,0.5,1.0,0.8,1.0,0.3,0.720,Mechanical Engineer; Maintenance Engineer; Manufacturing Engineer
9,STU-SAMPLE-001,Pendidikan Dokter,PRG2-016,Farmasi,0.3,1.0,1.0,0.5,0.5,0.8,0.5,0.3,0.645,Pharmacist Assistant Pathway; QA Analyst; Regulatory Affairs


Saved: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\reports\stage09\sample_scoring_simulation_v2.csv


## Markdown Cell — 14. Rancangan Output Chatbot Production-Ready

Output chatbot v2 perlu lebih transparan, tidak hanya menampilkan daftar prodi. Struktur respons yang disarankan:

1. **Ringkasan Profil Siswa**
   - jenis sekolah;
   - jurusan/konsentrasi asal;
   - minat dominan;
   - kecerdasan dominan;
   - gaya belajar;
   - grit/mindset;
   - career alignment.

2. **Hasil Perbandingan Pilihan Awal**
   - pilihan awal siswa;
   - status: selaras / cukup selaras / perlu eksplorasi alternatif;
   - alasan berbasis score component.

3. **Top-5 Rekomendasi Program Studi**
   - nama prodi;
   - skor akhir;
   - alasan utama;
   - komponen yang kuat;
   - komponen yang perlu diperkuat;
   - prospek karier.

4. **Roadmap Belajar Awal**
   - 0–1 bulan: eksplorasi dan validasi minat;
   - 2–3 bulan: dasar akademik dan skill awal;
   - 4–6 bulan: project/portofolio sederhana.

5. **Catatan Etis dan Akademik**
   - rekomendasi bersifat awal;
   - keputusan akhir perlu mempertimbangkan nilai akademik, biaya, lokasi kampus, akreditasi, konsultasi guru BK/orang tua, dan kondisi pribadi siswa.

In [18]:
# 18. Template Output Chatbot Production-Ready v2

top5 = simulation_df.head(5)
initial_choice_row = simulation_df[simulation_df['program_name'].str.lower() == sample_student_profile['initial_choice_program'].lower()]
initial_score = None if initial_choice_row.empty else float(initial_choice_row['final_score'].iloc[0])
initial_status = 'Belum ditemukan pada taxonomy' if initial_score is None else ('Selaras kuat' if initial_score >= 0.80 else 'Cukup selaras' if initial_score >= 0.65 else 'Perlu eksplorasi alternatif')

recommendation_lines = []
for i, row in top5.iterrows():
    recommendation_lines.append(
        f"{i+1}. {row['program_name']} — skor {row['final_score']}\n"
        f"   Alasan: cocok pada komponen RIASEC={row['riasec_fit']}, MI={row['multiple_intelligence_fit']}, "
        f"school fit={row['school_background_fit']}, career fit={row['career_alignment_fit']}.\n"
        f"   Prospek: {row['career_examples']}"
    )

chatbot_output_blueprint = f"""
# Contoh Rancangan Respons Chatbot v2

Halo, saya bantu petakan profil awal kamu berdasarkan informasi yang kamu berikan.

## 1. Ringkasan Profil
- Sekolah/Jurusan asal: {sample_student_profile['school_type']} - {sample_student_profile['school_major_detail']}
- Minat dominan/RIASEC: {sample_student_profile['riasec_primary']} dan {sample_student_profile['riasec_secondary']}
- Kecerdasan dominan: {sample_student_profile['mi_primary']}
- Gaya belajar: {sample_student_profile['vark_primary']}
- Grit/Mindset: {sample_student_profile['grit_level']} / {sample_student_profile['mindset_type']}
- Career alignment: {sample_student_profile['career_alignment']}

## 2. Perbandingan dengan Pilihan Awal
Pilihan awal kamu: **{sample_student_profile['initial_choice_program']}**  
Status keselarasan: **{initial_status}**

Catatan: pilihan awal tetap boleh dieksplorasi, tetapi sistem menemukan beberapa alternatif yang lebih selaras dengan sinyal coding, data, matematika, dan latar RPL.

## 3. Top-5 Rekomendasi Program Studi
{chr(10).join(recommendation_lines)}

## 4. Roadmap Awal 3 Bulan
- Bulan 1: validasi minat melalui mini project dan eksplorasi mata kuliah dasar.
- Bulan 2: pelajari dasar Matematika, Logika, dan skill teknis sesuai prodi prioritas.
- Bulan 3: buat portofolio sederhana dan konsultasikan hasilnya dengan guru BK/orang tua/pembimbing.

## 5. Catatan
Rekomendasi ini bersifat awal, bukan keputusan final. Pertimbangkan nilai akademik, minat jangka panjang, biaya, lokasi kampus, akreditasi, dan konsultasi dengan guru BK/orang tua.
""".strip()

blueprint_output = REPORT_STAGE09_DIR / 'sample_chatbot_output_blueprint_v2.md'
with open(blueprint_output, 'w', encoding='utf-8') as f:
    f.write(chatbot_output_blueprint)

display(Markdown(chatbot_output_blueprint))
print('Saved:', blueprint_output)

# Contoh Rancangan Respons Chatbot v2

Halo, saya bantu petakan profil awal kamu berdasarkan informasi yang kamu berikan.

## 1. Ringkasan Profil
- Sekolah/Jurusan asal: SMK - RPL
- Minat dominan/RIASEC: Investigative dan Realistic
- Kecerdasan dominan: Logis-Matematis
- Gaya belajar: Kinesthetic
- Grit/Mindset: Medium / Growth
- Career alignment: Praktikal

## 2. Perbandingan dengan Pilihan Awal
Pilihan awal kamu: **Pendidikan Dokter**  
Status keselarasan: **Selaras kuat**

Catatan: pilihan awal tetap boleh dieksplorasi, tetapi sistem menemukan beberapa alternatif yang lebih selaras dengan sinyal coding, data, matematika, dan latar RPL.

## 3. Top-5 Rekomendasi Program Studi
1. Teknik Informatika — skor 0.945
   Alasan: cocok pada komponen RIASEC=1.0, MI=1.0, school fit=1.0, career fit=1.0.
   Prospek: Software Engineer; Backend Developer; AI Engineer
2. Statistika — skor 0.87
   Alasan: cocok pada komponen RIASEC=1.0, MI=1.0, school fit=1.0, career fit=0.5.
   Prospek: Statistician; Data Analyst; Survey Researcher
3. Sains Data — skor 0.87
   Alasan: cocok pada komponen RIASEC=1.0, MI=1.0, school fit=1.0, career fit=0.5.
   Prospek: Data Analyst; Data Scientist; ML Engineer
4. Pendidikan Dokter — skor 0.855
   Alasan: cocok pada komponen RIASEC=1.0, MI=1.0, school fit=0.5, career fit=0.5.
   Prospek: Dokter; Clinical Researcher; Medical Educator
5. Kimia — skor 0.845
   Alasan: cocok pada komponen RIASEC=1.0, MI=1.0, school fit=0.5, career fit=0.5.
   Prospek: Chemist; Lab Analyst; Quality Control Analyst

## 4. Roadmap Awal 3 Bulan
- Bulan 1: validasi minat melalui mini project dan eksplorasi mata kuliah dasar.
- Bulan 2: pelajari dasar Matematika, Logika, dan skill teknis sesuai prodi prioritas.
- Bulan 3: buat portofolio sederhana dan konsultasikan hasilnya dengan guru BK/orang tua/pembimbing.

## 5. Catatan
Rekomendasi ini bersifat awal, bukan keputusan final. Pertimbangkan nilai akademik, minat jangka panjang, biaya, lokasi kampus, akreditasi, dan konsultasi dengan guru BK/orang tua.

Saved: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\reports\stage09\sample_chatbot_output_blueprint_v2.md


## Markdown Cell — 15. Rancangan Struktur Folder Output Tahap 09

Setelah notebook dijalankan, file output yang diharapkan:

```text
data/
  taxonomy/
    school_taxonomy_v2.csv
    program_studi_taxonomy_pddikti_style_v2.csv
    riasec_taxonomy_v2.csv
    multiple_intelligences_taxonomy_v2.csv
    vark_taxonomy_v2.csv
    grit_mindset_taxonomy_v2.csv
    career_alignment_taxonomy_v2.csv
models/
  recommender_scoring_config_v2.json
reports/
  stage09/
    current_project_inventory_stage09.csv
    gap_analysis_v1_to_v2.csv
    functional_requirements_v2.csv
    non_functional_requirements_v2.csv
    user_persona_v2.csv
    taxonomy_data_dictionary_v2.csv
    dataset_schema_v2.csv
    sample_scoring_simulation_v2.csv
    sample_chatbot_output_blueprint_v2.md
    stage09_output_checklist.csv
    stage09_summary.md
```

In [19]:
# 19. Checklist Output Tahap 09

expected_outputs = [
    TAXONOMY_DIR / 'school_taxonomy_v2.csv',
    TAXONOMY_DIR / 'program_studi_taxonomy_pddikti_style_v2.csv',
    TAXONOMY_DIR / 'riasec_taxonomy_v2.csv',
    TAXONOMY_DIR / 'multiple_intelligences_taxonomy_v2.csv',
    TAXONOMY_DIR / 'vark_taxonomy_v2.csv',
    TAXONOMY_DIR / 'grit_mindset_taxonomy_v2.csv',
    TAXONOMY_DIR / 'career_alignment_taxonomy_v2.csv',
    MODEL_DIR / 'recommender_scoring_config_v2.json',
    REPORT_STAGE09_DIR / 'current_project_inventory_stage09.csv',
    REPORT_STAGE09_DIR / 'gap_analysis_v1_to_v2.csv',
    REPORT_STAGE09_DIR / 'functional_requirements_v2.csv',
    REPORT_STAGE09_DIR / 'non_functional_requirements_v2.csv',
    REPORT_STAGE09_DIR / 'user_persona_v2.csv',
    REPORT_STAGE09_DIR / 'taxonomy_data_dictionary_v2.csv',
    REPORT_STAGE09_DIR / 'dataset_schema_v2.csv',
    REPORT_STAGE09_DIR / 'sample_scoring_simulation_v2.csv',
    REPORT_STAGE09_DIR / 'sample_chatbot_output_blueprint_v2.md',
]

checklist_df = pd.DataFrame([
    {
        'output_file': str(p.relative_to(PROJECT_ROOT)) if str(p).startswith(str(PROJECT_ROOT)) else str(p),
        'exists': p.exists(),
        'size_bytes': p.stat().st_size if p.exists() else 0
    }
    for p in expected_outputs
])

checklist_output = REPORT_STAGE09_DIR / 'stage09_output_checklist.csv'
checklist_df.to_csv(checklist_output, index=False)

display(checklist_df)
print('Saved:', checklist_output)

,output_file,exists,size_bytes
0,data\taxonomy\school_taxonomy_v2.csv,True,5175
1,data\taxonomy\program_studi_taxonomy_pddikti_style_v2.csv,True,10786
2,data\taxonomy\riasec_taxonomy_v2.csv,True,1386
3,data\taxonomy\multiple_intelligences_taxonomy_v2.csv,True,1550
4,data\taxonomy\vark_taxonomy_v2.csv,True,1027
5,data\taxonomy\grit_mindset_taxonomy_v2.csv,True,1523
6,data\taxonomy\career_alignment_taxonomy_v2.csv,True,1216
7,models\recommender_scoring_config_v2.json,True,996
8,reports\stage09\current_project_inventory_stage09.csv,True,2686
9,reports\stage09\gap_analysis_v1_to_v2.csv,True,2437


Saved: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\reports\stage09\stage09_output_checklist.csv


## 16. Kesimpulan Akademik

Tahap 09 menunjukkan bahwa pengembangan EduPath Career Assistant dari v1 ke v2 membutuhkan redesign yang tidak hanya menyentuh model machine learning, tetapi juga requirement, taxonomy, skema data, dan explainability. Berdasarkan evaluasi v1, sistem sudah berjalan end-to-end, tetapi masih memiliki keterbatasan pada ukuran dataset, cakupan program studi, representasi latar belakang siswa, dan ketahanan intent routing terhadap input ambigu.

Redesign v2 memperkenalkan taxonomy yang lebih sesuai dengan konteks Indonesia, yaitu jenis sekolah SMA/MA/SMK/MAK, jurusan atau konsentrasi siswa, katalog program studi S1 bergaya PDDikti, serta pemetaan profil berbasis RIASEC, Multiple Intelligences, VARK, Grit/Mindset, dan Career Alignment. Dengan pendekatan ini, rekomendasi program studi tidak hanya berbasis keyword, tetapi juga mempertimbangkan kesesuaian minat, kecerdasan dominan, gaya belajar, ketahanan belajar, latar jurusan sekolah, orientasi karier, dan pilihan awal siswa.

Dari sisi Data Mining, tahap ini memperkuat fase business understanding, data understanding, dan data preparation dalam CRISP-DM. Output tahap ini menjadi fondasi untuk tahap berikutnya, yaitu synthetic data generation, perluasan intent dataset, pembuatan student profile dataset, retraining intent classifier, dan pengembangan hybrid recommender system yang lebih explainable. Dengan demikian, EduPath Career Assistant v2 lebih layak diposisikan sebagai prototype production-oriented yang dapat divalidasi secara akademik, teknis, dan praktis.

In [21]:
# 20. Simpan Ringkasan Tahap 09

summary_text = f"""
# TAHAP 09 — Requirement & Taxonomy Redesign for Production-Ready v2

## Ringkasan
Tahap 09 berhasil menyusun requirement dan taxonomy redesign untuk EduPath Career Assistant v2.
Fokus tahap ini adalah memindahkan sistem dari baseline akademik menuju prototype production-oriented.

## Output Utama
1. Gap analysis dari v1 ke v2.
2. Functional requirement dan non-functional requirement.
3. Redesign user persona.
4. Taxonomy sekolah Indonesia: SMA/MA/SMK/MAK.
5. Taxonomy program studi S1 PDDikti-style sebanyak {len(program_taxonomy)} program.
6. Taxonomy RIASEC, Multiple Intelligences, VARK, Grit/Mindset, dan Career Alignment.
7. Data dictionary seluruh taxonomy.
8. Rancangan struktur dataset v2.
9. Konfigurasi awal scoring engine hybrid recommender.
10. Blueprint output chatbot production-ready.

## Catatan Validasi
- Field kode PDDikti belum diisi agar tidak terjadi klaim kode prodi yang belum tervalidasi.
- Taxonomy SMK/MAK masih subset awal dan perlu diperluas dari sumber spektrum keahlian resmi.
- Skor recommender masih rancangan awal dan perlu dikalibrasi dengan domain expert.

## Arah Tahap Berikutnya
Tahap 10 disarankan berfokus pada Synthetic Data Generator v2, perluasan dataset intent, pembuatan synthetic student profiles, dan evaluasi kualitas data synthetic.
""".strip()

summary_output = REPORT_STAGE09_DIR / 'stage09_summary.md'
with open(summary_output, 'w', encoding='utf-8') as f:
    f.write(summary_text)

display(Markdown(summary_text))
print('Saved:', summary_output)

# TAHAP 09 — Requirement & Taxonomy Redesign for Production-Ready v2

## Ringkasan
Tahap 09 berhasil menyusun requirement dan taxonomy redesign untuk EduPath Career Assistant v2.
Fokus tahap ini adalah memindahkan sistem dari baseline akademik menuju prototype production-oriented.

## Output Utama
1. Gap analysis dari v1 ke v2.
2. Functional requirement dan non-functional requirement.
3. Redesign user persona.
4. Taxonomy sekolah Indonesia: SMA/MA/SMK/MAK.
5. Taxonomy program studi S1 PDDikti-style sebanyak 35 program.
6. Taxonomy RIASEC, Multiple Intelligences, VARK, Grit/Mindset, dan Career Alignment.
7. Data dictionary seluruh taxonomy.
8. Rancangan struktur dataset v2.
9. Konfigurasi awal scoring engine hybrid recommender.
10. Blueprint output chatbot production-ready.

## Catatan Validasi
- Field kode PDDikti belum diisi agar tidak terjadi klaim kode prodi yang belum tervalidasi.
- Taxonomy SMK/MAK masih subset awal dan perlu diperluas dari sumber spektrum keahlian resmi.
- Skor recommender masih rancangan awal dan perlu dikalibrasi dengan domain expert.

## Arah Tahap Berikutnya
Tahap 10 disarankan berfokus pada Synthetic Data Generator v2, perluasan dataset intent, pembuatan synthetic student profiles, dan evaluasi kualitas data synthetic.

Saved: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\reports\stage09\stage09_summary.md
